In [2]:
import os
import zipfile
from tqdm import tqdm  # 可选，用于显示压缩进度

def compress_to_zip(source_path, output_zip_path, include_root=True):
    """
    压缩文件/文件夹到 ZIP 包
    :param source_path: 待压缩的文件/文件夹路径（如 autodl-tmp/cnap_1s_sv/final_model1012/resnet_LSTM_SE/20251204_152309）
    :param output_zip_path: 输出 ZIP 文件路径（如 compressed_file.zip）
    :param include_root: 压缩时是否包含根文件夹（True=包含，False=仅压缩文件夹内内容）
    """
    # 检查源路径是否存在
    if not os.path.exists(source_path):
        raise FileNotFoundError(f"源路径不存在: {source_path}")

    # 创建 ZIP 文件对象（压缩模式，允许大文件）
    with zipfile.ZipFile(output_zip_path, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as zipf:
        # 判断是文件还是文件夹
        if os.path.isfile(source_path):
            # 压缩单个文件
            file_name = os.path.basename(source_path)
            zipf.write(source_path, file_name)
            print(f"已压缩单个文件: {file_name}")
        else:
            # 压缩文件夹（遍历所有子文件/子文件夹）
            file_list = []
            for root, dirs, files in os.walk(source_path):
                for file in files:
                    file_path = os.path.join(root, file)
                    file_list.append(file_path)
            
            # 带进度条压缩
            for file_path in tqdm(file_list, desc="压缩进度"):
                # 计算压缩包内的相对路径
                if include_root:
                    arcname = os.path.relpath(file_path, os.path.dirname(source_path))
                else:
                    arcname = os.path.relpath(file_path, source_path)
                zipf.write(file_path, arcname)
    
    # 验证压缩结果
    if os.path.exists(output_zip_path):
        zip_size = os.path.getsize(output_zip_path) / (1024 * 1024)  # 转换为 MB
        print(f"\n压缩完成！ZIP 文件路径: {output_zip_path}")
        print(f"ZIP 文件大小: {zip_size:.2f} MB")
    else:
        raise RuntimeError("压缩失败，未生成 ZIP 文件")

if __name__ == "__main__":
    # 配置路径（根据实际需求修改）
    SOURCE_PATH = "/root/autodl-tmp/cnap_1s_sv/final_model1012/resnet_LSTM_SE/20251204_152309"  # 待压缩路径
    OUTPUT_ZIP = "/root/autodl-tmp/cnap_1s_sv/final_model1012/resnet_LSTM_SE/compressed_20251204_152309.zip"  # 输出 ZIP 路径
    
    # 安装进度条依赖（如果未安装）
    try:
        from tqdm import tqdm
    except ImportError:
        os.system("pip install tqdm")
        from tqdm import tqdm
    
    # 执行压缩
    try:
        compress_to_zip(
            source_path=SOURCE_PATH,
            output_zip_path=OUTPUT_ZIP,
            include_root=True  # 保留原文件夹结构
        )
    except Exception as e:
        print(f"压缩出错: {str(e)}")

压缩进度: 100%|██████████| 27/27 [00:08<00:00,  3.04it/s] 


压缩完成！ZIP 文件路径: /root/autodl-tmp/cnap_1s_sv/final_model1012/resnet_LSTM_SE/compressed_20251204_152309.zip
ZIP 文件大小: 164.76 MB


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, BatchNormalization, Activation, Add, GlobalAveragePooling1D, Dense, Dropout, LSTM, Multiply, Reshape
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr
from datetime import datetime
from sklearn.preprocessing import StandardScaler
import logging
import joblib
import tf2onnx  # 新增：导入tf2onnx库
import onnx  # 新增：导入onnx库

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.FileHandler('final_model_training.log'), logging.StreamHandler()]
)

# Set font
plt.rcParams["font.family"] = ["Arial", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

# ================== Basic Utility Functions ==================
def set_seed(seed=42):
    """Set random seeds for reproducibility"""
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'

def read_csv_with_encoding(file_path):
    encodings = ['utf-8', 'gbk', 'latin-1', 'utf-16']
    for encoding in encodings:
        try:
            df = pd.read_csv(
                file_path, 
                encoding=encoding,
                sep=',',
                on_bad_lines='skip'
            )
            logging.info(f"Successfully read file using encoding: {encoding} (sep=',' engine='c')")
            return df
        except:
            try:
                df = pd.read_csv(
                    file_path, 
                    encoding=encoding,
                    sep=None,
                    engine='python',
                    on_bad_lines='skip'
                )
                logging.info(f"Successfully read file using encoding: {encoding} (auto-sep engine='python')")
                return df
            except Exception as e:
                logging.warning(f"Failed to read with encoding {encoding}: {str(e)}")
    logging.error(f"All encodings failed for file: {file_path}")
    return None

def calculate_derivatives(signal):
    """Calculate first and second derivatives of the signal"""
    if len(signal.shape) == 3:
        signal = signal.reshape(signal.shape[0], signal.shape[1])
    
    first_deriv = np.zeros_like(signal)
    second_deriv = np.zeros_like(signal)
    
    # First derivative
    first_deriv[:, 1:-1] = (signal[:, 2:] - signal[:, :-2]) / 2
    first_deriv[:, 0] = signal[:, 1] - signal[:, 0]
    first_deriv[:, -1] = signal[:, -1] - signal[:, -2]
    
    # Second derivative
    second_deriv[:, 1:-1] = (first_deriv[:, 2:] - first_deriv[:, :-2]) / 2
    second_deriv[:, 0] = first_deriv[:, 1] - first_deriv[:, 0]
    second_deriv[:, -1] = first_deriv[:, -1] - first_deriv[:, -2]
    
    return first_deriv, second_deriv

def augment_signal(signal, prob=0.3):
    """修复信号增强函数中的维度不匹配问题"""
    augmented = signal.copy()
    # 1. 随机时间偏移（±2个点）
    if np.random.random() < prob:
        shift = np.random.randint(-2, 3)
        augmented = np.roll(augmented, shift, axis=0)
        
        if shift > 0:
            augmented[:shift, :] = augmented[shift:shift+1, :]
        elif shift < 0:
            augmented[shift:, :] = augmented[shift-1:shift, :]
    
    # 2. 加轻微高斯噪声（噪声幅度为信号标准差的5%）
    if np.random.random() < prob:
        noise = np.random.normal(0, 0.05 * np.std(augmented, axis=0, keepdims=True), size=augmented.shape)
        augmented += noise
    
    return augmented

def extract_data(df):
    """Extract features and target values"""
    required_cols = ['time', 'age', 'gender', 'weight', 'height', 'BSA', 'BMI', 'HR', 'SV']
    signal_cols = [f'signal_{i}' for i in range(1, 126)]
    required_cols.extend(signal_cols)
    
    df = df[required_cols].copy()
    df = df.dropna(subset=['SV'])
    df = df.fillna(df.mean(numeric_only=True))
    
    time = df['time'].values
    indices = df.index.values
    
    age = df['age'].values
    gender = df['gender'].values
    weight = df['weight'].values
    height = df['height'].values
    bsa = df['BSA'].values
    bmi = df['BMI'].values
    hr = df['HR'].values
    
    raw_signal = df[signal_cols].values
    first_deriv, second_deriv = calculate_derivatives(raw_signal)
    
    # 3-channel input (num_samples, 125, 3)
    X = np.stack([raw_signal, first_deriv, second_deriv], axis=-1)
    
    y_sv = df['SV'].values
    y_co = (y_sv * hr) / 1000
    y = np.column_stack((y_sv, y_co))
    
    logging.info(f"Feature extraction completed - Input feature shape: {X.shape}")
    return X, y, age, gender, weight, height, bsa, bmi, hr, time, indices, df

def split_data_by_time_groups(X, y, age, gender, weight, height, bsa, bmi, hr, sv, time, indices,
                             val_ratio=0.2, test_ratio=0.2, random_state=42):
    """Split dataset by time groups"""
    unique_times = np.unique(time)
    n_groups = len(unique_times)
    logging.info(f"Detected {n_groups} time groups")
    
    if n_groups < 3:
        raise ValueError(f"Too few time groups ({n_groups} groups), at least 3 groups required")
    
    all_indices = np.arange(n_groups)
    total_test_groups = int(np.ceil(n_groups * test_ratio))
    total_val_groups = int(np.ceil((n_groups - total_test_groups) * val_ratio))
    
    if total_test_groups + total_val_groups >= n_groups:
        logging.warning("Insufficient groups, adjusting split ratio")
        total_test_groups = max(1, int(n_groups * 0.2))
        total_val_groups = max(1, int((n_groups - total_test_groups) * 0.2))
    
    from sklearn.model_selection import train_test_split
    train_val_indices, test_indices = train_test_split(
        all_indices, test_size=total_test_groups, random_state=random_state
    )
    train_indices, val_indices = train_test_split(
        train_val_indices, test_size=total_val_groups, random_state=random_state
    )
    
    train_times = unique_times[train_indices]
    val_times = unique_times[val_indices]
    test_times = unique_times[test_indices]
    
    train_mask = np.isin(time, train_times)
    val_mask = np.isin(time, val_times)
    test_mask = np.isin(time, test_times)
    
    def extract(mask):
        return (
            X[mask], y[mask], age[mask], gender[mask], weight[mask],
            height[mask], bsa[mask], bmi[mask], hr[mask], time[mask], indices[mask]
        )
    
    X_train, y_train, age_train, gender_train, weight_train, height_train, bsa_train, bmi_train, hr_train, time_train, indices_train = extract(train_mask)
    X_val, y_val, age_val, gender_val, weight_val, height_val, bsa_val, bmi_val, hr_val, time_val, indices_val = extract(val_mask)
    X_test, y_test, age_test, gender_test, weight_test, height_test, bsa_test, bmi_test, hr_test, time_test, indices_test = extract(test_mask)
    
    logging.info(f"Dataset split completed - Training set: {len(X_train)} samples, Validation set: {len(X_val)} samples, Test set: {len(X_test)} samples")
    logging.info(f"Training set contains {len(train_indices)} time groups, Validation set contains {len(val_indices)} time groups, Test set contains {len(test_indices)} time groups")
    
    return (X_train, X_val, X_test, y_train, y_val, y_test,
            age_train, age_val, age_test, gender_train, gender_val, gender_test,
            weight_train, weight_val, weight_test, height_train, height_val, height_test,
            bsa_train, bsa_val, bsa_test, bmi_train, bmi_val, bmi_test,
            hr_train, hr_val, hr_test, time_train, time_val, time_test,
            indices_train, indices_val, indices_test)

def standardize_features(train, val, test, is_2d=False):
    """Standardize features"""
    scaler = StandardScaler()
    if is_2d:
        train_reshaped = train.reshape(train.shape[0], -1)
        scaler.fit(train_reshaped)
        return (
            scaler.transform(train_reshaped).reshape(train.shape),
            scaler.transform(val.reshape(val.shape[0], -1)).reshape(val.shape),
            scaler.transform(test.reshape(test.shape[0], -1)).reshape(test.shape),
            scaler
        )
    else:
        train_reshaped = train.reshape(-1, 1)
        scaler.fit(train_reshaped)
        return (
            scaler.transform(train_reshaped).flatten(),
            scaler.transform(val.reshape(-1, 1)).flatten(),
            scaler.transform(test.reshape(-1, 1)).flatten(),
            scaler
        )

# ================== 新增：生成预测结果的函数 ==================
def generate_predictions(model, X_std, age_std, gender_std, weight_std, height_std, bsa_std, bmi_std, hr_std, hr_original):
    """
    生成SV（模型预测）和CO（基于SV预测值+原始HR计算）的预测结果
    """
    # 预测SV（模型输出）
    pred_sv = model.predict(
        [X_std, age_std, gender_std, weight_std, height_std, bsa_std, bmi_std, hr_std],
        verbose=0
    ).flatten()
    
    # 计算CO（CO = SV * HR / 1000，HR用原始值避免标准化误差）
    pred_co = (pred_sv * hr_original) / 1000
    
    return pred_sv, pred_co

# ================== 修改：保存数据集（新增预测结果列） ==================
def save_datasets_with_predictions(
    save_dir, 
    # 原始数据集特征
    X_train, X_val, X_test, y_train, y_val, y_test,
    age_train, age_val, age_test, gender_train, gender_val, gender_test,
    weight_train, weight_val, weight_test, height_train, height_val, height_test,
    bsa_train, bsa_val, bsa_test, bmi_train, bmi_val, bmi_test,
    hr_train, hr_val, hr_test, time_train, time_val, time_test,
    indices_train, indices_val, indices_test,
    # 预测结果
    pred_sv_train=None, pred_co_train=None,
    pred_sv_val=None, pred_co_val=None,
    pred_sv_test=None, pred_co_test=None
):
    """
    保存训练/验证/测试集，包含原始特征+真实值+预测值
    """
    data_dir = os.path.join(save_dir, "datasets")
    os.makedirs(data_dir, exist_ok=True)
    
    # 训练集
    train_df = pd.DataFrame({
        # 原始临床特征
        'age': age_train, 'gender': gender_train, 'weight': weight_train,
        'height': height_train, 'BSA': bsa_train, 'BMI': bmi_train, 'HR': hr_train,
        # 时间与原始索引
        'time': time_train, 'original_index': indices_train,
        # 真实值（SV和CO）
        'true_sv': y_train[:, 0], 'true_co': y_train[:, 1]
    })
    # 添加训练集预测结果（如果提供）
    if pred_sv_train is not None and pred_co_train is not None:
        train_df['pred_sv'] = pred_sv_train
        train_df['pred_co'] = pred_co_train
    train_df.to_csv(os.path.join(data_dir, "train_set.csv"), index=False)
    
    # 验证集
    val_df = pd.DataFrame({
        'age': age_val, 'gender': gender_val, 'weight': weight_val,
        'height': height_val, 'BSA': bsa_val, 'BMI': bmi_val, 'HR': hr_val,
        'time': time_val, 'original_index': indices_val,
        'true_sv': y_val[:, 0], 'true_co': y_val[:, 1],
        # 验证集预测结果
        'pred_sv': pred_sv_val, 'pred_co': pred_co_val
    })
    val_df.to_csv(os.path.join(data_dir, "val_set.csv"), index=False)
    
    # 测试集
    test_df = pd.DataFrame({
        'age': age_test, 'gender': gender_test, 'weight': weight_test,
        'height': height_test, 'BSA': bsa_test, 'BMI': bmi_test, 'HR': hr_test,
        'time': time_test, 'original_index': indices_test,
        'true_sv': y_test[:, 0], 'true_co': y_test[:, 1],
        # 测试集预测结果
        'pred_sv': pred_sv_test, 'pred_co': pred_co_test
    })
    test_df.to_csv(os.path.join(data_dir, "test_set.csv"), index=False)
    
    # 保存原始信号特征
    np.save(os.path.join(data_dir, "X_train.npy"), X_train)
    np.save(os.path.join(data_dir, "X_val.npy"), X_val)
    np.save(os.path.join(data_dir, "X_test.npy"), X_test)
    
    logging.info(f"Datasets with predictions saved to: {data_dir}")

# ================== 新增：保存ONNX模型函数 ==================
def save_model_to_onnx(model, save_dir, model_name="resnet18_lstm_se_model.onnx"):
    """
    将TensorFlow模型转换并保存为ONNX格式
    """
    try:
        # 构建输入签名（匹配模型输入）
        input_signatures = [
            tf.TensorSpec((None, 125, 3), tf.float32, name="signal_input"),
            tf.TensorSpec((None, 1), tf.float32, name="age_input"),
            tf.TensorSpec((None, 1), tf.float32, name="gender_input"),
            tf.TensorSpec((None, 1), tf.float32, name="weight_input"),
            tf.TensorSpec((None, 1), tf.float32, name="height_input"),
            tf.TensorSpec((None, 1), tf.float32, name="bsa_input"),
            tf.TensorSpec((None, 1), tf.float32, name="bmi_input"),
            tf.TensorSpec((None, 1), tf.float32, name="hr_input")
        ]
        
        # 转换模型为ONNX格式
        onnx_model, _ = tf2onnx.convert.from_keras(
            model,
            input_signature=input_signatures,
            opset=13,  # ONNX算子集版本
            output_path=os.path.join(save_dir, model_name)
        )
        
        # 验证ONNX模型
        onnx.checker.check_model(onnx_model)
        logging.info(f"ONNX模型已成功保存至：{os.path.join(save_dir, model_name)}")
        return True
    except Exception as e:
        logging.error(f"保存ONNX模型失败：{str(e)}")
        return False

# ================== SE Block Definition ==================
class SEBlock(tf.keras.layers.Layer):
    """Squeeze-and-Excitation Block for channel attention"""
    def __init__(self, reduction_ratio=8, **kwargs):
        super().__init__(** kwargs)
        self.reduction_ratio = reduction_ratio

    def build(self, input_shape):
        channels = input_shape[-1]
        self.squeeze = GlobalAveragePooling1D()
        self.excitation1 = Dense(channels // self.reduction_ratio, activation='relu')
        self.excitation2 = Dense(channels, activation='sigmoid')
        super().build(input_shape)

    def call(self, inputs):
        x = self.squeeze(inputs)
        x = self.excitation1(x)
        x = self.excitation2(x)
        x = Reshape((1, -1))(x)
        return Multiply()([inputs, x])

# ================== Model Definition with LSTM and SE ==================
class ResidualSEBlock(tf.keras.layers.Layer):
    """Residual block with SE attention mechanism"""
    def __init__(self, filters, stride=1, use_1x1_conv=False, kernel_regularizer=None, **kwargs):
        super().__init__(** kwargs)
        self.conv1 = Conv1D(filters, 3, padding='same', strides=stride, kernel_regularizer=kernel_regularizer)
        self.bn1 = BatchNormalization()
        self.act1 = Activation('relu')
        self.conv2 = Conv1D(filters, 3, padding='same', kernel_regularizer=kernel_regularizer)
        self.bn2 = BatchNormalization()
        self.se = SEBlock()
        
        if use_1x1_conv:
            self.shortcut_conv = Conv1D(filters, 1, strides=stride, kernel_regularizer=kernel_regularizer)
            self.shortcut_bn = BatchNormalization()
        self.use_1x1_conv = use_1x1_conv
        self.act2 = Activation('relu')

    def call(self, inputs):
        x = self.act1(self.bn1(self.conv1(inputs)))
        x = self.bn2(self.conv2(x))
        x = self.se(x)
        
        if self.use_1x1_conv:
            shortcut = self.shortcut_bn(self.shortcut_conv(inputs))
        else:
            shortcut = inputs
        return self.act2(x + shortcut)

def build_resnet18_with_lstm_se(input_shape):
    """Build ResNet18 model with LSTM and SE modules"""
    best_hp = {
        "lr": 0.0008,
        "l2_reg": 4.114391549630821e-05,
        "dropout": 0.3,
        "dense_units": 128,
        "lstm_units": 64
    }
    
    regularizer = l2(best_hp["l2_reg"]) if best_hp["l2_reg"] > 0 else None
    
    inputs = Input(shape=input_shape, name="signal_input")
    
    # 先卷积提取局部特征
    x = Conv1D(64, 7, strides=2, padding='same', kernel_regularizer=regularizer)(inputs)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    # 再用LSTM捕捉时序依赖
    x_lstm = LSTM(best_hp["lstm_units"], return_sequences=True, kernel_regularizer=regularizer)(x)
    
    # ResNet18 with SE blocks
    for _ in range(2):
        x = ResidualSEBlock(64, kernel_regularizer=regularizer)(x_lstm)
    
    x = ResidualSEBlock(128, stride=2, use_1x1_conv=True, kernel_regularizer=regularizer)(x)
    for _ in range(1):
        x = ResidualSEBlock(128, kernel_regularizer=regularizer)(x)
    
    x = ResidualSEBlock(256, stride=2, use_1x1_conv=True, kernel_regularizer=regularizer)(x)
    for _ in range(1):
        x = ResidualSEBlock(256, kernel_regularizer=regularizer)(x)
    
    x = ResidualSEBlock(512, stride=2, use_1x1_conv=True, kernel_regularizer=regularizer)(x)
    for _ in range(1):
        x = ResidualSEBlock(512, kernel_regularizer=regularizer)(x)
    
    x = GlobalAveragePooling1D()(x)
    x = Dense(best_hp["dense_units"], activation='relu', kernel_regularizer=regularizer)(x)
    x = Dropout(best_hp["dropout"])(x)
    
    # Clinical feature inputs
    age_in = Input(shape=(1,), name="age_input")
    gender_in = Input(shape=(1,), name="gender_input")
    weight_in = Input(shape=(1,), name="weight_input")
    height_in = Input(shape=(1,), name="height_input")
    bsa_in = Input(shape=(1,), name="bsa_input")
    bmi_in = Input(shape=(1,), name="bmi_input")
    hr_in = Input(shape=(1,), name="hr_input")
    
    x = tf.keras.layers.Concatenate()([x, age_in, gender_in, weight_in, height_in, bsa_in, bmi_in, hr_in])
    outputs = Dense(1, name="sv_output")(x)
    
    model = Model(
        inputs=[inputs, age_in, gender_in, weight_in, height_in, bsa_in, bmi_in, hr_in],
        outputs=outputs
    )
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=best_hp["lr"], 
            clipvalue=1.0,
            beta_1=0.9,
            beta_2=0.999,
            epsilon=1e-07
        ),
        loss=tf.keras.losses.Huber(delta=1.0),
        metrics=['mae']
    )
    
    return model, best_hp

# ================== Model Evaluation ==================
def evaluate_final_model(model, X, y_true, hr, time_groups, dataset_name, save_dir):
    """Evaluate model and save results"""
    y_pred_sv = model.predict(X, verbose=0).flatten()
    
    if len(y_pred_sv) != len(y_true[:, 0]):
        min_length = min(len(y_pred_sv), len(y_true[:, 0]))
        y_pred_sv = y_pred_sv[:min_length]
        y_true = y_true[:min_length]
        hr = hr[:min_length]
        time_groups = time_groups[:min_length]
        logging.warning(f"Prediction and true value lengths do not match, truncated to {min_length} samples")
    
    y_pred_co = (y_pred_sv * hr) / 1000
    y_true_sv = y_true[:, 0]
    y_true_co = y_true[:, 1]
    
    def calculate_metrics(y_true, y_pred, name):
        mse = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)
        corr, p_val = pearsonr(y_true, y_pred)
        
        logging.info(f"\n{name} evaluation metrics:")
        logging.info(f"MSE: {mse:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f}")
        logging.info(f"R²: {r2:.4f} | Correlation coefficient: {corr:.4f} (p={p_val:.4e})")
        
        return {
            "mse": mse, "rmse": rmse, "mae": mae,
            "r2": r2, "corr": corr, "p_value": p_val,
            "y_true": y_true, "y_pred": y_pred
        }
    
    sv_metrics = calculate_metrics(y_true_sv, y_pred_sv, f"{dataset_name} - SV")
    co_metrics = calculate_metrics(y_true_co, y_pred_co, f"{dataset_name} - CO (calculated)")
    
    # 误差分布分析
    plt.figure(figsize=(12, 5))
    for i, (item, title) in enumerate([
        (sv_metrics, f"{dataset_name} SV Error Distribution"),
        (co_metrics, f"{dataset_name} CO Error Distribution")
    ]):
        plt.subplot(1, 2, i+1)
        error = item["y_pred"] - item["y_true"]
        plt.hist(error, bins=30, alpha=0.7)
        plt.axvline(x=0, color='r', linestyle='--')
        plt.xlabel("Prediction Error")
        plt.ylabel("Count")
        plt.title(title)
        plt.grid(ls='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(f"{save_dir}/{dataset_name}_error_distribution.png", dpi=300)
    plt.close()
    
    # 预测值vs真实值
    plt.figure(figsize=(12, 5))
    for i, (item, title) in enumerate([
        (sv_metrics, f"{dataset_name} SV Predictions vs True Values (R²={sv_metrics['r2']:.2f})"),
        (co_metrics, f"{dataset_name} CO Predictions vs True Values (R²={co_metrics['r2']:.2f})")
    ]):
        plt.subplot(1, 2, i+1)
        true_vals = item["y_true"]
        pred_vals = item["y_pred"]
        min_len = min(len(true_vals), len(pred_vals))
        
        plt.scatter(true_vals[:min_len], pred_vals[:min_len], alpha=0.6, s=30)
        min_val = min(true_vals[:min_len].min(), pred_vals[:min_len].min())
        max_val = max(true_vals[:min_len].max(), pred_vals[:min_len].max())
        plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1)
        plt.xlabel("True Value")
        plt.ylabel("Predicted Value")
        plt.title(title)
        plt.grid(ls='--', alpha=0.5)
    
    plt.tight_layout()
    plt.savefig(f"{save_dir}/{dataset_name}_pred_vs_true.png", dpi=300)
    plt.close()
    
    # 按时间组分析
    time_groups_unique = np.unique(time_groups)
    group_metrics = []
    for t in time_groups_unique:
        mask = time_groups == t
        if np.sum(mask) < 5:
            continue
        group_r2 = r2_score(y_true_sv[mask], y_pred_sv[mask])
        group_metrics.append({"time_group": t, "sample_count": np.sum(mask), "r2": group_r2})
    
    if group_metrics:
        pd.DataFrame(group_metrics).to_csv(f"{save_dir}/{dataset_name}_time_group_metrics.csv", index=False)
        logging.info(f"Saved time group metrics for {dataset_name} with {len(group_metrics)} groups")
    
    if dataset_name == "Test set":
        analyze_grouped_predictions(y_true_sv, y_pred_sv, y_true_co, y_pred_co, save_dir)
    
    return {"SV": sv_metrics, "CO": co_metrics}

def analyze_grouped_predictions(y_true_sv, y_pred_sv, y_true_co, y_pred_co, save_dir):
    """Grouped analysis for test set"""
    df_sv = pd.DataFrame({'true_value': y_true_sv, 'pred_value': y_pred_sv})
    df_co = pd.DataFrame({'true_value': y_true_co, 'pred_value': y_pred_co})
    
    grouped_sv = df_sv.groupby(df_sv['true_value'].round(1)).agg({'pred_value': 'mean'}).reset_index()
    grouped_co = df_co.groupby(df_co['true_value'].round(1)).agg({'pred_value': 'mean'}).reset_index()
    
    def calculate_grouped_metrics(grouped_df, name):
        mse = mean_squared_error(grouped_df['true_value'], grouped_df['pred_value'])
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(grouped_df['true_value'], grouped_df['pred_value'])
        r2 = r2_score(grouped_df['true_value'], grouped_df['pred_value'])
        corr, p_val = pearsonr(grouped_df['true_value'], grouped_df['pred_value'])
        
        logging.info(f"\nGrouped {name} evaluation metrics:")
        logging.info(f"MSE: {mse:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f}")
        logging.info(f"R²: {r2:.4f} | Correlation coefficient: {corr:.4f} (p={p_val:.4e})")
        
        return {"mse": mse, "rmse": rmse, "mae": mae, "r2": r2, "corr": corr, "p_value": p_val, "grouped_data": grouped_df}
    
    grouped_sv_metrics = calculate_grouped_metrics(grouped_sv, "SV")
    grouped_co_metrics = calculate_grouped_metrics(grouped_co, "CO")
    
    grouped_sv.to_csv(f"{save_dir}/test_grouped_sv.csv", index=False)
    grouped_co.to_csv(f"{save_dir}/test_grouped_co.csv", index=False)
    
    plt.figure(figsize=(12, 5))
    for i, (grouped_data, title, metric) in enumerate([
        (grouped_sv, f"Test Set Grouped SV Prediction Mean vs True Values (R²={grouped_sv_metrics['r2']:.2f})", grouped_sv_metrics),
        (grouped_co, f"Test Set Grouped CO Prediction Mean vs True Values (R²={grouped_co_metrics['r2']:.2f})", grouped_co_metrics)
    ]):
        plt.subplot(1, 2, i+1)
        plt.scatter(grouped_data['true_value'], grouped_data['pred_value'], alpha=0.7, s=50, c='green')
        min_val = min(grouped_data['true_value'].min(), grouped_data['pred_value'].min())
        max_val = max(grouped_data['true_value'].max(), grouped_data['pred_value'].max())
        plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1)
        
        plt.text(0.05, 0.95, 
                 f"r = {metric['corr']:.3f}\nn = {len(grouped_data)}", 
                 transform=plt.gca().transAxes,
                 verticalalignment='top',
                 bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        plt.xlabel("True Value")
        plt.ylabel("Mean Predicted Value")
        plt.title(title)
        plt.grid(ls='--', alpha=0.5)
    
    plt.tight_layout()
    plt.savefig(f"{save_dir}/test_grouped_pred_vs_true.png", dpi=300)
    plt.close()

# ================== Main Function ==================
def main():
    set_seed()
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    save_dir = f"D:/心输出量项目/CNAP脉氧处理数据/脉氧+CNAP/resnet_LSTM_SE/{timestamp}"
    os.makedirs(save_dir, exist_ok=True)
    logging.info(f"All results will be saved to: {save_dir}")
    
    # 1. 读取数据
    file_path = "D:/心输出量项目/CNAP脉氧处理数据/脉氧+CNAP/11.17-11.21脉氧+CNAP.csv"
    df = read_csv_with_encoding(file_path)
    if df is None or len(df) == 0:
        logging.error("Could not read valid data, program terminated")
        return
    
    # 2. 提取特征
    try:
        X, y, age, gender, weight, height, bsa, bmi, hr, time, indices, original_df = extract_data(df)
    except Exception as e:
        logging.error(f"Data extraction failed: {e}")
        return
    
    # 异常值处理
    sv_values = y[:, 0]
    q1, q3 = np.percentile(sv_values, [25, 75])
    iqr = q3 - q1
    upper_bound = q3 + 1.5 * iqr
    valid_mask = sv_values <= upper_bound
    
    X = X[valid_mask]
    y = y[valid_mask]
    age = age[valid_mask]
    gender = gender[valid_mask]
    weight = weight[valid_mask]
    height = height[valid_mask]
    bsa = bsa[valid_mask]
    bmi = bmi[valid_mask]
    hr = hr[valid_mask]
    time = time[valid_mask]
    indices = indices[valid_mask]
    original_df = original_df[valid_mask].copy()
    
    removed_count = len(valid_mask) - sum(valid_mask)
    logging.info(f"使用IQR法检测异常值，上界={upper_bound:.2f}，移除{removed_count}个样本，剩余有效样本: {len(X)}")
    
    if len(X) < 100:
        logging.error(f"Insufficient data volume after filtering (only {len(X)} samples), cannot train model")
        return
    
    # 3. 划分数据集
    try:
        split_result = split_data_by_time_groups(
            X, y, age, gender, weight, height, bsa, bmi, hr, y[:, 0], time, indices
        )
        (X_train, X_val, X_test, y_train, y_val, y_test,
         age_train, age_val, age_test, gender_train, gender_val, gender_test,
         weight_train, weight_val, weight_test, height_train, height_val, height_test,
         bsa_train, bsa_val, bsa_test, bmi_train, bmi_val, bmi_test,
         hr_train, hr_val, hr_test, time_train, time_val, time_test,
         indices_train, indices_val, indices_test) = split_result
    except Exception as e:
        logging.error(f"Dataset splitting failed: {e}")
        return
    
    # 训练集数据增强
    X_train_augmented = np.array([augment_signal(x) for x in X_train])
    logging.info(f"Applied data augmentation to training set: {X_train_augmented.shape}")
    
    # 4. 标准化特征
    logging.info("Standardizing features...")
    X_train_std, X_val_std, X_test_std, x_scaler = standardize_features(X_train_augmented, X_val, X_test, is_2d=True)
    
    age_train_std, age_val_std, age_test_std, _ = standardize_features(age_train, age_val, age_test)
    gender_train_std, gender_val_std, gender_test_std = gender_train, gender_val, gender_test
    weight_train_std, weight_val_std, weight_test_std, _ = standardize_features(weight_train, weight_val, weight_test)
    height_train_std, height_val_std, height_test_std, _ = standardize_features(height_train, height_val, height_test)
    bsa_train_std, bsa_val_std, bsa_test_std, _ = standardize_features(bsa_train, bsa_val, bsa_test)
    bmi_train_std, bmi_val_std, bmi_test_std, _ = standardize_features(bmi_train, bmi_val, bmi_test)
    hr_train_std, hr_val_std, hr_test_std, _ = standardize_features(hr_train, hr_val, hr_test)
    
    joblib.dump(x_scaler, f"{save_dir}/signal_scaler.pkl")
    logging.info("Feature scaler saved")
    
    # 5. 构建模型
    logging.info("Building ResNet18 model with LSTM and SE modules...")
    model, best_hp = build_resnet18_with_lstm_se(input_shape=(125, 3))
    model.summary(print_fn=lambda x: logging.info(x))
    
    with open(f"{save_dir}/best_hyperparameters.txt", "w") as f:
        for key, value in best_hp.items():
            f.write(f"{key}: {value}\n")
    
    # 6. 训练模型
    logging.info("Starting model training...")
    batch_size = 32 if len(X_train) < 1000 else 64
    logging.info(f"Using batch size: {batch_size} based on training set size")
    
    def cosine_annealing(epoch, lr):
        initial_lr = best_hp["lr"]
        epochs = 50
        return initial_lr * (1 + np.cos(np.pi * epoch / epochs)) / 2
    
    callbacks = [
        LearningRateScheduler(cosine_annealing, verbose=1)
    ]
    
    history = model.fit(
        [X_train_std, age_train_std, gender_train_std, weight_train_std, 
         height_train_std, bsa_train_std, bmi_train_std, hr_train_std],
        y_train[:, 0],
        epochs=50,
        batch_size=batch_size,
        validation_data=(
            [X_val_std, age_val_std, gender_val_std, weight_val_std, 
             height_val_std, bsa_val_std, bmi_val_std, hr_val_std],
            y_val[:, 0]
        ),
        callbacks=callbacks,
        verbose=1
    )
    
    pd.DataFrame(history.history).to_csv(f"{save_dir}/training_history.csv", index=False)
    
    # 绘制训练曲线
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Loss Curves')
    plt.xlabel('Epoch')
    plt.ylabel('Huber Loss')
    plt.legend()
    plt.grid(ls='--', alpha=0.5)
    
    plt.subplot(1, 2, 2)
    plt.plot(history.history['mae'], label='Training MAE')
    plt.plot(history.history['val_mae'], label='Validation MAE')
    plt.title('MAE Curves')
    plt.xlabel('Epoch')
    plt.ylabel('MAE')
    plt.legend()
    plt.grid(ls='--', alpha=0.5)
    
    plt.tight_layout()
    plt.savefig(f"{save_dir}/training_curves.png", dpi=300)
    plt.close()
    
    # 7. 生成预测结果
    logging.info("Generating predictions for all datasets...")
    # 训练集预测
    pred_sv_train, pred_co_train = generate_predictions(
        model, X_train_std, age_train_std, gender_train_std, weight_train_std, 
        height_train_std, bsa_train_std, bmi_train_std, hr_train_std, hr_train
    )
    
    # 验证集预测
    pred_sv_val, pred_co_val = generate_predictions(
        model, X_val_std, age_val_std, gender_val_std, weight_val_std, 
        height_val_std, bsa_val_std, bmi_val_std, hr_val_std, hr_val
    )
    
    # 测试集预测
    pred_sv_test, pred_co_test = generate_predictions(
        model, X_test_std, age_test_std, gender_test_std, weight_test_std, 
        height_test_std, bsa_test_std, bmi_test_std, hr_test_std, hr_test
    )
    
    # 8. 保存包含预测结果的数据集
    save_datasets_with_predictions(
        save_dir,
        X_train, X_val, X_test, y_train, y_val, y_test,
        age_train, age_val, age_test, gender_train, gender_val, gender_test,
        weight_train, weight_val, weight_test, height_train, height_val, height_test,
        bsa_train, bsa_val, bsa_test, bmi_train, bmi_val, bmi_test,
        hr_train, hr_val, hr_test, time_train, time_val, time_test,
        indices_train, indices_val, indices_test,
        pred_sv_train, pred_co_train,
        pred_sv_val, pred_co_val,
        pred_sv_test, pred_co_test
    )
    
    # 9. 评估模型（修复NameError：将评估逻辑移到main函数内部，确保model变量可见）
    logging.info("\n===== Starting model evaluation =====")
    test_data = [X_test_std, age_test_std, gender_test_std, weight_test_std, 
                 height_test_std, bsa_test_std, bmi_test_std, hr_test_std]
    val_data = [X_val_std, age_val_std, gender_val_std, weight_val_std, 
                height_val_std, bsa_val_std, bmi_val_std, hr_val_std]
    train_data = [X_train_std, age_train_std, gender_train_std, weight_train_std, 
                  height_train_std, bsa_train_std, bmi_train_std, hr_train_std]
    
    evaluate_final_model(model, train_data, y_train, hr_train, time_train, "Training set", save_dir)
    evaluate_final_model(model, val_data, y_val, hr_val, time_val, "Validation set", save_dir)
    test_metrics = evaluate_final_model(model, test_data, y_test, hr_test, time_test, "Test set", save_dir)
    
    # 10. 保存模型（包含ONNX格式）
    # 保存Keras原生模型
    model.save(f"{save_dir}/resnet18_lstm_se_model")
    logging.info(f"ResNet18 with LSTM and SE model saved to: {save_dir}/resnet18_lstm_se_model")
    
    # 保存ONNX格式模型
    save_model_to_onnx(model, save_dir, "resnet18_lstm_se_model.onnx")
    
    logging.info("\nAll processes completed!")

if __name__ == "__main__":
    main()

D:\PythonProject\PythonProject3\.venv\Lib\site-packages\tf2onnx\utils.py:49: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  onnx_pb.TensorProto.STRING: np.object,


AttributeError: module 'numpy' has no attribute 'object'.
`np.object` was a deprecated alias for the builtin `object`. To avoid this error in existing code, use `object` by itself. Doing this will not modify any behavior and is safe. 
The aliases was originally deprecated in NumPy 1.20; for more details and guidance see the original release note at:
    https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations

In [ ]:
构建无血压的推理模型

In [5]:
import os
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
import onnxruntime as ort
from sklearn.preprocessing import StandardScaler
import logging

# ================== 配置日志 ==================
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler()]
)

# ================== 自定义层定义（加载Keras模型必需） ==================
class SEBlock(tf.keras.layers.Layer):
    """Squeeze-and-Excitation Block"""
    def __init__(self, reduction_ratio=8, **kwargs):
        super().__init__(** kwargs)
        self.reduction_ratio = reduction_ratio

    def build(self, input_shape):
        channels = input_shape[-1]
        self.squeeze = tf.keras.layers.GlobalAveragePooling1D()
        self.excitation1 = tf.keras.layers.Dense(channels // self.reduction_ratio, activation='relu')
        self.excitation2 = tf.keras.layers.Dense(channels, activation='sigmoid')
        super().build(input_shape)

    def call(self, inputs):
        x = self.squeeze(inputs)
        x = self.excitation1(x)
        x = self.excitation2(x)
        x = tf.keras.layers.Reshape((1, -1))(x)
        return tf.keras.layers.Multiply()([inputs, x])

class ResidualSEBlock(tf.keras.layers.Layer):
    """Residual block with SE attention"""
    def __init__(self, filters, stride=1, use_1x1_conv=False, kernel_regularizer=None, **kwargs):
        super().__init__(** kwargs)
        self.conv1 = tf.keras.layers.Conv1D(filters, 3, padding='same', strides=stride, kernel_regularizer=kernel_regularizer)
        self.bn1 = tf.keras.layers.BatchNormalization()
        self.act1 = tf.keras.layers.Activation('relu')
        self.conv2 = tf.keras.layers.Conv1D(filters, 3, padding='same', strides=stride, kernel_regularizer=kernel_regularizer)
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.se = SEBlock()
        
        if use_1x1_conv:
            self.shortcut_conv = tf.keras.layers.Conv1D(filters, 1, strides=stride, kernel_regularizer=kernel_regularizer)
            self.shortcut_bn = tf.keras.layers.BatchNormalization()
        self.use_1x1_conv = use_1x1_conv
        self.act2 = tf.keras.layers.Activation('relu')

    def call(self, inputs):
        x = self.act1(self.bn1(self.conv1(inputs)))
        x = self.bn2(self.conv2(x))
        x = self.se(x)
        
        if self.use_1x1_conv:
            shortcut = self.shortcut_bn(self.shortcut_conv(inputs))
        else:
            shortcut = inputs
        return self.act2(x + shortcut)

# ================== 推理模型类（修复维度错误） ==================
class SVCOInferenceModel:
    def __init__(self, scaler_path, model_dir):
        """
        初始化推理模型（指定标准化器路径）
        :param scaler_path: signal_scaler.pkl的完整路径
        :param model_dir: 模型文件所在目录
        """
        self.scaler_path = scaler_path  # 单独指定标准化器路径
        self.model_dir = model_dir
        self.model_type = None
        self.model = None
        self.signal_scaler = None
        self._load_scaler()

    def _load_scaler(self):
        """加载指定路径的信号标准化器"""
        if not os.path.exists(self.scaler_path):
            raise FileNotFoundError(f"信号标准化器不存在: {self.scaler_path}")
        self.signal_scaler = joblib.load(self.scaler_path)
        logging.info(f"✅ 信号标准化器加载完成（路径：{self.scaler_path}）")

    def load_keras_model(self):
        """加载Keras模型"""
        model_path = os.path.join(self.model_dir, "resnet18_lstm_se_model")
        if not os.path.exists(model_path):
            raise FileNotFoundError(f"Keras模型不存在: {model_path}")
        
        self.model = tf.keras.models.load_model(
            model_path, 
            custom_objects={'SEBlock': SEBlock, 'ResidualSEBlock': ResidualSEBlock}
        )
        self.model_type = "keras"
        logging.info(f"✅ Keras模型加载完成（路径：{model_path}）")

    def load_onnx_model(self):
        """加载ONNX模型"""
        onnx_path = os.path.join(self.model_dir, "resnet18_lstm_se_model.onnx")
        if not os.path.exists(onnx_path):
            raise FileNotFoundError(f"ONNX模型不存在: {onnx_path}")
        
        providers = ['CPUExecutionProvider']
        if ort.get_device() == "GPU":
            providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
        
        self.model = ort.InferenceSession(onnx_path, providers=providers)
        self.model_type = "onnx"
        logging.info(f"✅ ONNX模型加载完成（路径：{onnx_path}，运行设备: {providers[0]}）")

    def _calculate_derivatives(self, signal):
        """
        修复维度错误：适配1维信号输入
        :param signal: 1维数组 (125,)
        :return: first_deriv (125,), second_deriv (125,)
        """
        # 确保信号是1维数组
        signal = np.squeeze(signal)
        if len(signal.shape) != 1 or signal.shape[0] != 125:
            raise ValueError(f"信号维度错误！需为125维，当前shape: {signal.shape}")
        
        first_deriv = np.zeros_like(signal)
        second_deriv = np.zeros_like(signal)
        
        # 一阶导数（1维索引）
        first_deriv[1:-1] = (signal[2:] - signal[:-2]) / 2
        first_deriv[0] = signal[1] - signal[0]
        first_deriv[-1] = signal[-1] - signal[-2]
        
        # 二阶导数（1维索引）
        second_deriv[1:-1] = (first_deriv[2:] - first_deriv[:-2]) / 2
        second_deriv[0] = first_deriv[1] - first_deriv[0]
        second_deriv[-1] = first_deriv[-1] - first_deriv[-2]
        
        return first_deriv, second_deriv

    def preprocess_single_sample(self, raw_signal, clinical_features):
        """
        预处理单个样本（修复维度错误）
        :param raw_signal: 1维数组 (125,)
        :param clinical_features: 列表 [age, gender, weight, height, BSA, BMI, HR]
        """
        # 1. 生成3通道信号（125, 3）
        first_deriv, second_deriv = self._calculate_derivatives(raw_signal)
        signal_3d = np.stack([raw_signal, first_deriv, second_deriv], axis=-1)  # (125, 3)
        
        # 2. 信号标准化（使用指定路径的scaler）
        # scaler期望输入是 (n_samples, n_features)，所以先reshape为(1, 375)（125*3）
        signal_3d_flat = signal_3d.reshape(1, -1)  # (1, 375)
        signal_3d_std_flat = self.signal_scaler.transform(signal_3d_flat)
        signal_3d_std = signal_3d_std_flat.reshape(1, 125, 3)  # (1, 125, 3)
        
        # 3. 临床特征格式整理
        clinical_arr = np.array(clinical_features).reshape(7, 1)  # (7,1)
        
        # 4. 整理模型输入
        inputs = [
            signal_3d_std,  # 信号输入 (1, 125, 3)
            np.array([[clinical_arr[0][0]]]),  # age (1,1)
            np.array([[clinical_arr[1][0]]]),  # gender (1,1)
            np.array([[clinical_arr[2][0]]]),  # weight (1,1)
            np.array([[clinical_arr[3][0]]]),  # height (1,1)
            np.array([[clinical_arr[4][0]]]),  # bsa (1,1)
            np.array([[clinical_arr[5][0]]]),  # bmi (1,1)
            np.array([[clinical_arr[6][0]]])   # hr (1,1)
        ]
        
        return inputs

    def infer_single(self, raw_signal, clinical_features):
        """单样本推理"""
        if self.model is None:
            raise RuntimeError("模型未加载！请先调用 load_keras_model() 或 load_onnx_model()")
        
        # 预处理
        inputs = self.preprocess_single_sample(raw_signal, clinical_features)
        
        # 推理
        if self.model_type == "keras":
            pred_sv = self.model.predict(inputs, verbose=0)[0][0]
        elif self.model_type == "onnx":
            onnx_inputs = {
                "signal_input": inputs[0].astype(np.float32),
                "age_input": inputs[1].astype(np.float32),
                "gender_input": inputs[2].astype(np.float32),
                "weight_input": inputs[3].astype(np.float32),
                "height_input": inputs[4].astype(np.float32),
                "bsa_input": inputs[5].astype(np.float32),
                "bmi_input": inputs[6].astype(np.float32),
                "hr_input": inputs[7].astype(np.float32)
            }
            pred_sv = self.model.run(None, onnx_inputs)[0][0][0]
        
        # 计算CO
        hr = clinical_features[6]
        pred_co = (pred_sv * hr) / 1000
        
        return {
            "sv": round(float(pred_sv), 2),
            "co": round(float(pred_co), 2)
        }

# ================== 加载测试集并推理 ==================
def load_test_data(test_csv_path, x_test_path):
    """加载测试集数据（指定具体路径）"""
    # 1. 加载test_set.csv
    if not os.path.exists(test_csv_path):
        raise FileNotFoundError(f"测试集CSV不存在: {test_csv_path}")
    test_df = pd.read_csv(test_csv_path)
    logging.info(f"✅ 加载测试集CSV，共 {len(test_df)} 个样本")
    
    # 2. 加载X_test.npy
    if not os.path.exists(x_test_path):
        raise FileNotFoundError(f"测试集信号文件不存在: {x_test_path}")
    X_test = np.load(x_test_path)
    logging.info(f"✅ 加载测试集信号，形状: {X_test.shape}")
    
    # 3. 校验样本数量一致性
    if len(test_df) != len(X_test):
        raise ValueError(f"样本数量不匹配！CSV: {len(test_df)}, 信号: {len(X_test)}")
    
    # 4. 校验信号维度
    if X_test.shape[1:] != (125, 3):
        raise ValueError(f"信号维度错误！需为(125,3)，当前: {X_test.shape[1:]}")
    
    return test_df, X_test

def run_test_set_inference(scaler_path, model_dir, test_csv_path, x_test_path):
    """运行测试集推理并对比结果"""
    # 1. 初始化推理模型（指定标准化器路径）
    infer_model = SVCOInferenceModel(
        scaler_path=scaler_path,
        model_dir=model_dir
    )
    
    # 2. 加载模型（二选一）
    infer_model.load_keras_model()  # 优先测试Keras模型
    # infer_model.load_onnx_model()  # 可选：测试ONNX模型
    
    # 3. 加载测试集数据
    test_df, X_test = load_test_data(test_csv_path, x_test_path)
    
    # 4. 批量推理
    infer_sv_list = []
    infer_co_list = []
    logging.info("\n开始测试集推理...")
    
    for idx in range(len(test_df)):
        # 提取单个样本数据（1维原始信号）
        raw_signal = X_test[idx, :, 0]  # 原始信号（第0通道），shape=(125,)
        clinical_features = [
            test_df.iloc[idx]['age'],
            test_df.iloc[idx]['gender'],
            test_df.iloc[idx]['weight'],
            test_df.iloc[idx]['height'],
            test_df.iloc[idx]['BSA'],
            test_df.iloc[idx]['BMI'],
            test_df.iloc[idx]['HR']
        ]
        
        # 推理
        result = infer_model.infer_single(raw_signal, clinical_features)
        infer_sv_list.append(result['sv'])
        infer_co_list.append(result['co'])
        
        # 进度打印
        if (idx + 1) % 50 == 0:
            logging.info(f"进度: {idx+1}/{len(test_df)} 样本已推理")
    
    # 5. 整理结果并对比
    test_df['infer_sv'] = infer_sv_list
    test_df['infer_co'] = infer_co_list
    
    # 计算推理结果与训练时预测值的差异
    test_df['sv_diff'] = abs(test_df['infer_sv'] - test_df['pred_sv'])
    test_df['co_diff'] = abs(test_df['infer_co'] - test_df['pred_co'])
    
    # 统计差异
    avg_sv_diff = test_df['sv_diff'].mean()
    avg_co_diff = test_df['co_diff'].mean()
    max_sv_diff = test_df['sv_diff'].max()
    max_co_diff = test_df['co_diff'].max()
    
    logging.info("\n================ 推理结果对比 ================")
    logging.info(f"训练时预测SV vs 推理模型SV：平均绝对差 = {avg_sv_diff:.2f} mL，最大绝对差 = {max_sv_diff:.2f} mL")
    logging.info(f"训练时预测CO vs 推理模型CO：平均绝对差 = {avg_co_diff:.2f} L/min，最大绝对差 = {max_co_diff:.2f} L/min")
    
    # 6. 保存推理结果
    result_save_path = os.path.join(model_dir, "test_set_infer_result.csv")
    test_df.to_csv(result_save_path, index=False)
    logging.info(f"\n✅ 推理结果已保存至：{result_save_path}")
    
    # 7. 打印前5个样本的对比示例
    logging.info("\n前5个样本对比示例：")
    compare_cols = ['true_sv', 'pred_sv', 'infer_sv', 'true_co', 'pred_co', 'infer_co']
    print(test_df[compare_cols].head())

# ================== 主函数（配置所有路径） ==================
if __name__ == "__main__":
    # ================== 核心路径配置（修正后） ==================
    # 1. 标准化器路径（你指定的路径）
    SCALER_PATH = "autodl-tmp/cnap_1s_sv/final_model1012/resnet_LSTM_SE/20251204_152309/signal_scaler.pkl"
    
    # 2. 模型目录（包含keras/onnx模型文件）
    MODEL_DIR = "autodl-tmp/cnap_1s_sv/final_model1012/resnet_LSTM_SE/20251204_152309"
    
    # 3. 测试集CSV路径（和信号同目录）
    TEST_CSV_PATH = "autodl-tmp/cnap_1s_sv/final_model1012/resnet_LSTM_SE/20251204_152309/datasets/test_set.csv"
    
    # 4. 测试集信号文件路径（和CSV同目录）
    X_TEST_PATH = "autodl-tmp/cnap_1s_sv/final_model1012/resnet_LSTM_SE/20251204_152309/datasets/X_test.npy"
    
    # ================== 执行推理 ==================
    try:
        run_test_set_inference(
            scaler_path=SCALER_PATH,
            model_dir=MODEL_DIR,
            test_csv_path=TEST_CSV_PATH,
            x_test_path=X_TEST_PATH
        )
    except Exception as e:
        logging.error(f"推理过程出错: {str(e)}")
        raise

2025-12-04 18:29:49,869 - INFO - ✅ 信号标准化器加载完成（路径：autodl-tmp/cnap_1s_sv/final_model1012/resnet_LSTM_SE/20251204_152309/signal_scaler.pkl）
2025-12-04 18:29:53,520 - INFO - ✅ Keras模型加载完成（路径：autodl-tmp/cnap_1s_sv/final_model1012/resnet_LSTM_SE/20251204_152309/resnet18_lstm_se_model）
2025-12-04 18:29:53,532 - INFO - ✅ 加载测试集CSV，共 10055 个样本
2025-12-04 18:29:53,547 - INFO - ✅ 加载测试集信号，形状: (10055, 125, 3)
2025-12-04 18:29:53,547 - INFO - 
开始测试集推理...
2025-12-04 18:29:53.575096: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:176] None of the MLIR Optimization Passes are enabled (registered 2)
2025-12-04 18:29:53.597563: I tensorflow/core/platform/profile_utils/cpu_utils.cc:114] CPU Frequency: 2100000000 Hz
2025-12-04 18:29:53.983678: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcudnn.so.8
2025-12-04 18:29:54.634767: I tensorflow/stream_executor/cuda/cuda_dnn.cc:359] Loaded cuDNN version 8101
2025-12-04 18:29:55.426160: W tensorf

   true_sv    pred_sv  infer_sv  true_co   pred_co  infer_co
0     89.0  92.358986    541.12    7.387  7.665796     44.91
1     89.0  94.613580    538.89    7.832  8.325995     47.42
2     89.0  92.604225    536.88    7.832  8.149172     47.25
3     89.0  87.549576    531.83    7.832  7.704363     46.80
4     89.0  94.622444    538.90    7.832  8.326775     47.42


In [ ]:
加入血压

In [7]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, BatchNormalization, Activation, Add, GlobalAveragePooling1D, Dense, Dropout, LSTM, Multiply, Reshape
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import LearningRateScheduler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr
from datetime import datetime
from sklearn.preprocessing import StandardScaler
import logging
import joblib
# 新增：ONNX导出依赖
import tf2onnx
import onnx
from onnxruntime import InferenceSession

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.FileHandler('final_model_training.log'), logging.StreamHandler()]
)

# Set font
plt.rcParams["font.family"] = ["Arial", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

# ================== Basic Utility Functions ==================
def set_seed(seed=42):
    """Set random seeds for reproducibility"""
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'

def read_csv_with_encoding(file_path):
    encodings = ['utf-8', 'gbk', 'latin-1', 'utf-16']
    for encoding in encodings:
        try:
            df = pd.read_csv(
                file_path, 
                encoding=encoding,
                sep=',',
                on_bad_lines='skip'
            )
            logging.info(f"Successfully read file using encoding: {encoding} (sep=',' engine='c')")
            return df
        except:
            try:
                df = pd.read_csv(
                    file_path, 
                    encoding=encoding,
                    sep=None,
                    engine='python',
                    on_bad_lines='skip'
                )
                logging.info(f"Successfully read file using encoding: {encoding} (auto-sep engine='python')")
                return df
            except Exception as e:
                logging.warning(f"Failed to read with encoding {encoding}: {str(e)}")
    logging.error(f"All encodings failed for file: {file_path}")
    return None

def calculate_derivatives(signal):
    """Calculate first and second derivatives of the signal"""
    if len(signal.shape) == 3:
        signal = signal.reshape(signal.shape[0], signal.shape[1])
    
    first_deriv = np.zeros_like(signal)
    second_deriv = np.zeros_like(signal)
    
    # First derivative
    first_deriv[:, 1:-1] = (signal[:, 2:] - signal[:, :-2]) / 2
    first_deriv[:, 0] = signal[:, 1] - signal[:, 0]
    first_deriv[:, -1] = signal[:, -1] - signal[:, -2]
    
    # Second derivative
    second_deriv[:, 1:-1] = (first_deriv[:, 2:] - first_deriv[:, :-2]) / 2
    second_deriv[:, 0] = first_deriv[:, 1] - first_deriv[:, 0]
    second_deriv[:, -1] = first_deriv[:, -1] - first_deriv[:, -2]
    
    return first_deriv, second_deriv

def augment_signal(signal, prob=0.3):
    """修复信号增强函数中的维度不匹配问题"""
    augmented = signal.copy()
    # 1. 随机时间偏移（±2个点）
    if np.random.random() < prob:
        shift = np.random.randint(-2, 3)
        augmented = np.roll(augmented, shift, axis=0)
        
        if shift > 0:
            augmented[:shift, :] = augmented[shift:shift+1, :]
        elif shift < 0:
            augmented[shift:, :] = augmented[shift-1:shift, :]
    
    # 2. 加轻微高斯噪声（噪声幅度为信号标准差的5%）
    if np.random.random() < prob:
        noise = np.random.normal(0, 0.05 * np.std(augmented, axis=0, keepdims=True), size=augmented.shape)
        augmented += noise
    
    return augmented

def extract_data(df):
    """Extract features and target values - 新增SBP、DBP、PP特征"""
    # 新增SBP、DBP到必需列列表
    required_cols = ['time', 'age', 'gender', 'weight', 'height', 'BSA', 'BMI', 'HR', 'SV', 'SBP', 'DBP']
    signal_cols = [f'signal_{i}' for i in range(1, 126)]
    required_cols.extend(signal_cols)
    
    df = df[required_cols].copy()
    # 新增SBP、DBP的缺失值处理
    df = df.dropna(subset=['SV', 'SBP', 'DBP'])
    df = df.fillna(df.mean(numeric_only=True))
    
    time = df['time'].values
    indices = df.index.values
    
    # 读取SBP、DBP原始值并计算PP
    age = df['age'].values
    gender = df['gender'].values
    weight = df['weight'].values
    height = df['height'].values
    bsa = df['BSA'].values
    bmi = df['BMI'].values
    hr = df['HR'].values
    sbp = df['SBP'].values  # 新增SBP
    dbp = df['DBP'].values  # 新增DBP
    pp = sbp - dbp          # 新增PP（脉压=收缩压-舒张压）
    
    raw_signal = df[signal_cols].values
    first_deriv, second_deriv = calculate_derivatives(raw_signal)
    
    # 3-channel input (num_samples, 125, 3)
    X = np.stack([raw_signal, first_deriv, second_deriv], axis=-1)
    
    y_sv = df['SV'].values
    y_co = (y_sv * hr) / 1000
    y = np.column_stack((y_sv, y_co))
    
    logging.info(f"Feature extraction completed - Input feature shape: {X.shape}")
    # 返回值中新增SBP、DBP、PP
    return X, y, age, gender, weight, height, bsa, bmi, hr, sbp, dbp, pp, time, indices, df

def split_data_by_time_groups(X, y, age, gender, weight, height, bsa, bmi, hr, sbp, dbp, pp, sv, time, indices,
                             val_ratio=0.2, test_ratio=0.2, random_state=42):
    """Split dataset by time groups - 新增SBP、DBP、PP参数传递"""
    unique_times = np.unique(time)
    n_groups = len(unique_times)
    logging.info(f"Detected {n_groups} time groups")
    
    if n_groups < 3:
        raise ValueError(f"Too few time groups ({n_groups} groups), at least 3 groups required")
    
    all_indices = np.arange(n_groups)
    total_test_groups = int(np.ceil(n_groups * test_ratio))
    total_val_groups = int(np.ceil((n_groups - total_test_groups) * val_ratio))
    
    if total_test_groups + total_val_groups >= n_groups:
        logging.warning("Insufficient groups, adjusting split ratio")
        total_test_groups = max(1, int(n_groups * 0.2))
        total_val_groups = max(1, int((n_groups - total_test_groups) * 0.2))
    
    from sklearn.model_selection import train_test_split
    train_val_indices, test_indices = train_test_split(
        all_indices, test_size=total_test_groups, random_state=random_state
    )
    train_indices, val_indices = train_test_split(
        train_val_indices, test_size=total_val_groups, random_state=random_state
    )
    
    train_times = unique_times[train_indices]
    val_times = unique_times[val_indices]
    test_times = unique_times[test_indices]
    
    train_mask = np.isin(time, train_times)
    val_mask = np.isin(time, val_times)
    test_mask = np.isin(time, test_times)
    
    def extract(mask):
        return (
            X[mask], y[mask], age[mask], gender[mask], weight[mask],
            height[mask], bsa[mask], bmi[mask], hr[mask], 
            sbp[mask], dbp[mask], pp[mask],  # 新增SBP、DBP、PP的掩码提取
            time[mask], indices[mask]
        )
    
    # 划分结果中新增SBP、DBP、PP的训练/验证/测试集
    X_train, y_train, age_train, gender_train, weight_train, height_train, bsa_train, bmi_train, hr_train, sbp_train, dbp_train, pp_train, time_train, indices_train = extract(train_mask)
    X_val, y_val, age_val, gender_val, weight_val, height_val, bsa_val, bmi_val, hr_val, sbp_val, dbp_val, pp_val, time_val, indices_val = extract(val_mask)
    X_test, y_test, age_test, gender_test, weight_test, height_test, bsa_test, bmi_test, hr_test, sbp_test, dbp_test, pp_test, time_test, indices_test = extract(test_mask)
    
    logging.info(f"Dataset split completed - Training set: {len(X_train)} samples, Validation set: {len(X_val)} samples, Test set: {len(X_test)} samples")
    logging.info(f"Training set contains {len(train_indices)} time groups, Validation set contains {len(val_indices)} time groups, Test set contains {len(test_indices)} time groups")
    
    # 返回值中新增SBP、DBP、PP的分组结果
    return (X_train, X_val, X_test, y_train, y_val, y_test,
            age_train, age_val, age_test, gender_train, gender_val, gender_test,
            weight_train, weight_val, weight_test, height_train, height_val, height_test,
            bsa_train, bsa_val, bsa_test, bmi_train, bmi_val, bmi_test,
            hr_train, hr_val, hr_test, sbp_train, sbp_val, sbp_test,
            dbp_train, dbp_val, dbp_test, pp_train, pp_val, pp_test,
            time_train, time_val, time_test, indices_train, indices_val, indices_test)

def standardize_features(train, val, test, is_2d=False):
    """Standardize features"""
    scaler = StandardScaler()
    if is_2d:
        train_reshaped = train.reshape(train.shape[0], -1)
        scaler.fit(train_reshaped)
        return (
            scaler.transform(train_reshaped).reshape(train.shape),
            scaler.transform(val.reshape(val.shape[0], -1)).reshape(val.shape),
            scaler.transform(test.reshape(test.shape[0], -1)).reshape(test.shape),
            scaler
        )
    else:
        train_reshaped = train.reshape(-1, 1)
        scaler.fit(train_reshaped)
        return (
            scaler.transform(train_reshaped).flatten(),
            scaler.transform(val.reshape(-1, 1)).flatten(),
            scaler.transform(test.reshape(-1, 1)).flatten(),
            scaler
        )

# ================== 生成预测结果的函数 ==================
def generate_predictions(model, X_std, age_std, gender_std, weight_std, height_std, bsa_std, bmi_std, hr_std, 
                         sbp_std, dbp_std, pp_std, hr_original):
    """
    生成SV（模型预测）和CO（基于SV预测值+原始HR计算）的预测结果
    新增SBP、DBP、PP标准化特征作为输入
    """
    # 预测SV（模型输出）
    pred_sv = model.predict(
        [X_std, age_std, gender_std, weight_std, height_std, bsa_std, bmi_std, hr_std,
         sbp_std, dbp_std, pp_std],  # 新增血压特征
        verbose=0
    ).flatten()
    
    # 计算CO（CO = SV * HR / 1000，HR用原始值避免标准化误差）
    pred_co = (pred_sv * hr_original) / 1000
    
    return pred_sv, pred_co

# ================== 保存数据集（包含预测结果列） ==================
def save_datasets_with_predictions(
    save_dir, 
    # 原始数据集特征
    X_train, X_val, X_test, y_train, y_val, y_test,
    age_train, age_val, age_test, gender_train, gender_val, gender_test,
    weight_train, weight_val, weight_test, height_train, height_val, height_test,
    bsa_train, bsa_val, bsa_test, bmi_train, bmi_val, bmi_test,
    hr_train, hr_val, hr_test, sbp_train, sbp_val, sbp_test,
    dbp_train, dbp_val, dbp_test, pp_train, pp_val, pp_test,
    time_train, time_val, time_test, indices_train, indices_val, indices_test,
    # 预测结果
    pred_sv_train=None, pred_co_train=None,
    pred_sv_val=None, pred_co_val=None,
    pred_sv_test=None, pred_co_test=None
):
    """
    保存训练/验证/测试集，包含原始特征+真实值+预测值
    新增SBP、DBP、PP特征的保存
    """
    data_dir = os.path.join(save_dir, "datasets")
    os.makedirs(data_dir, exist_ok=True)
    
    # 训练集
    train_df = pd.DataFrame({
        # 原始临床特征（新增血压相关）
        'age': age_train, 'gender': gender_train, 'weight': weight_train,
        'height': height_train, 'BSA': bsa_train, 'BMI': bmi_train, 'HR': hr_train,
        'SBP': sbp_train, 'DBP': dbp_train, 'PP': pp_train,
        # 时间与原始索引
        'time': time_train, 'original_index': indices_train,
        # 真实值（SV和CO）
        'true_sv': y_train[:, 0], 'true_co': y_train[:, 1]
    })
    # 添加训练集预测结果（如果提供）
    if pred_sv_train is not None and pred_co_train is not None:
        train_df['pred_sv'] = pred_sv_train
        train_df['pred_co'] = pred_co_train
    train_df.to_csv(os.path.join(data_dir, "train_set.csv"), index=False)
    
    # 验证集
    val_df = pd.DataFrame({
        'age': age_val, 'gender': gender_val, 'weight': weight_val,
        'height': height_val, 'BSA': bsa_val, 'BMI': bmi_val, 'HR': hr_val,
        'SBP': sbp_val, 'DBP': dbp_val, 'PP': pp_val,
        'time': time_val, 'original_index': indices_val,
        'true_sv': y_val[:, 0], 'true_co': y_val[:, 1],
        # 验证集预测结果
        'pred_sv': pred_sv_val, 'pred_co': pred_co_val
    })
    val_df.to_csv(os.path.join(data_dir, "val_set.csv"), index=False)
    
    # 测试集
    test_df = pd.DataFrame({
        'age': age_test, 'gender': gender_test, 'weight': weight_test,
        'height': height_test, 'BSA': bsa_test, 'BMI': bmi_test, 'HR': hr_test,
        'SBP': sbp_test, 'DBP': dbp_test, 'PP': pp_test,
        'time': time_test, 'original_index': indices_test,
        'true_sv': y_test[:, 0], 'true_co': y_test[:, 1],
        # 测试集预测结果
        'pred_sv': pred_sv_test, 'pred_co': pred_co_test
    })
    test_df.to_csv(os.path.join(data_dir, "test_set.csv"), index=False)
    
    # 保存原始信号特征
    np.save(os.path.join(data_dir, "X_train.npy"), X_train)
    np.save(os.path.join(data_dir, "X_val.npy"), X_val)
    np.save(os.path.join(data_dir, "X_test.npy"), X_test)
    
    logging.info(f"Datasets with predictions saved to: {data_dir}")

# ================== 新增：绘制Bland-Altman图（针对测试集） ==================
def plot_bland_altman(y_true, y_pred, metric_name, save_path):
    """
    绘制Bland-Altman图（一致性分析图）
    Args:
        y_true: 真实值（1D数组）
        y_pred: 预测值（1D数组）
        metric_name: 指标名称（如"SV"、"CO"）
        save_path: 图片保存路径
    Returns:
        一致性分析结果（平均偏差、95%一致性界限）
    """
    # 计算均值和偏差
    mean_vals = (y_true + y_pred) / 2  # 真实值与预测值的均值
    biases = y_pred - y_true            # 预测值 - 真实值（偏差）
    
    # 计算统计指标
    mean_bias = np.mean(biases)                # 平均偏差
    std_bias = np.std(biases, ddof=1)          # 偏差的标准差
    lower_limit = mean_bias - 1.96 * std_bias  # 95%一致性下界
    upper_limit = mean_bias + 1.96 * std_bias  # 95%一致性上界
    
    # 绘制图形
    plt.figure(figsize=(10, 6))
    plt.scatter(mean_vals, biases, alpha=0.6, s=40, color='#2E86AB', edgecolors='white', linewidth=0.5)
    
    # 绘制参考线（平均偏差、95%界限）
    plt.axhline(y=mean_bias, color='#A23B72', linestyle='-', linewidth=2, label=f'Mean Bias: {mean_bias:.2f}')
    plt.axhline(y=lower_limit, color='#F18F01', linestyle='--', linewidth=1.5, label=f'95% LoA: [{lower_limit:.2f}, {upper_limit:.2f}]')
    plt.axhline(y=upper_limit, color='#F18F01', linestyle='--', linewidth=1.5)
    plt.axhline(y=0, color='gray', linestyle=':', linewidth=1, alpha=0.8)
    
    # 添加标签和图例
    plt.xlabel(f'Mean of True and Predicted {metric_name}')
    plt.ylabel(f'Predicted - True {metric_name}')
    plt.title(f'Bland-Altman Plot for {metric_name} (Test Set)')
    plt.legend(loc='best')
    plt.grid(linestyle='--', alpha=0.6)
    plt.tight_layout()
    
    # 保存图片
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    # 返回统计结果
    return {
        'mean_bias': mean_bias,
        'lower_limit': lower_limit,
        'upper_limit': upper_limit,
        'std_bias': std_bias
    }

# ================== SE Block Definition ==================
class SEBlock(tf.keras.layers.Layer):
    """Squeeze-and-Excitation Block for channel attention"""
    def __init__(self, reduction_ratio=8, **kwargs):
        super().__init__(** kwargs)
        self.reduction_ratio = reduction_ratio

    def build(self, input_shape):
        channels = input_shape[-1]
        self.squeeze = GlobalAveragePooling1D()
        self.excitation1 = Dense(channels // self.reduction_ratio, activation='relu')
        self.excitation2 = Dense(channels, activation='sigmoid')
        super().build(input_shape)

    def call(self, inputs):
        x = self.squeeze(inputs)
        x = self.excitation1(x)
        x = self.excitation2(x)
        x = Reshape((1, -1))(x)
        return Multiply()([inputs, x])

# ================== Model Definition with ResNet+SE first, then LSTM ==================
class ResidualSEBlock(tf.keras.layers.Layer):
    """Residual block with SE attention mechanism"""
    def __init__(self, filters, stride=1, use_1x1_conv=False, kernel_regularizer=None, **kwargs):
        super().__init__(** kwargs)
        self.conv1 = Conv1D(filters, 3, padding='same', strides=stride, kernel_regularizer=kernel_regularizer)
        self.bn1 = BatchNormalization()
        self.act1 = Activation('relu')
        self.conv2 = Conv1D(filters, 3, padding='same', kernel_regularizer=kernel_regularizer)
        self.bn2 = BatchNormalization()
        self.se = SEBlock()
        
        if use_1x1_conv:
            self.shortcut_conv = Conv1D(filters, 1, strides=stride, kernel_regularizer=kernel_regularizer)
            self.shortcut_bn = BatchNormalization()
        self.use_1x1_conv = use_1x1_conv
        self.act2 = Activation('relu')

    def call(self, inputs):
        x = self.act1(self.bn1(self.conv1(inputs)))
        x = self.bn2(self.conv2(x))
        x = self.se(x)
        
        if self.use_1x1_conv:
            shortcut = self.shortcut_bn(self.shortcut_conv(inputs))
        else:
            shortcut = inputs
        return self.act2(x + shortcut)

def build_resnet18_with_se_lstm(input_shape):
    """Build model with ResNet+SE first, then LSTM - 新增血压特征输入"""
    best_hp = {
        "lr": 0.001,
        "l2_reg": 4.114391549630821e-05,
        "dropout": 0.3,
        "dense_units": 128,  # 增加 dense 单元以适应新增特征
        "lstm_units": 64
    }
    
    regularizer = l2(best_hp["l2_reg"]) if best_hp["l2_reg"] > 0 else None
    
    inputs = Input(shape=input_shape, name="signal_input")
    
    # 第一步：使用ResNet+SE提取局部特征和通道注意力
    x = Conv1D(64, 7, strides=2, padding='same', kernel_regularizer=regularizer)(inputs)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    # ResNet18 with SE blocks
    for _ in range(2):
        x = ResidualSEBlock(64, kernel_regularizer=regularizer)(x)
    
    x = ResidualSEBlock(128, stride=2, use_1x1_conv=True, kernel_regularizer=regularizer)(x)
    for _ in range(1):
        x = ResidualSEBlock(128, kernel_regularizer=regularizer)(x)
    
    x = ResidualSEBlock(256, stride=2, use_1x1_conv=True, kernel_regularizer=regularizer)(x)
    for _ in range(1):
        x = ResidualSEBlock(256, kernel_regularizer=regularizer)(x)
    
    # 减少一层ResNet以保留更多时序信息供LSTM处理
    x = ResidualSEBlock(512, stride=1, use_1x1_conv=True, kernel_regularizer=regularizer)(x)
    
    # 第二步：使用LSTM捕捉时序依赖关系
    x_lstm = LSTM(best_hp["lstm_units"], return_sequences=False, kernel_regularizer=regularizer)(x)
    
    x = Dense(best_hp["dense_units"], activation='relu', kernel_regularizer=regularizer)(x_lstm)
    x = Dropout(best_hp["dropout"])(x)
    
    # Clinical feature inputs - 新增SBP、DBP、PP输入
    age_in = Input(shape=(1,), name="age_input")
    gender_in = Input(shape=(1,), name="gender_input")
    weight_in = Input(shape=(1,), name="weight_input")
    height_in = Input(shape=(1,), name="height_input")
    bsa_in = Input(shape=(1,), name="bsa_input")
    bmi_in = Input(shape=(1,), name="bmi_input")
    hr_in = Input(shape=(1,), name="hr_input")
    sbp_in = Input(shape=(1,), name="sbp_input")  # 新增SBP输入
    dbp_in = Input(shape=(1,), name="dbp_input")  # 新增DBP输入
    pp_in = Input(shape=(1,), name="pp_input")    # 新增PP输入
    
    # 合并所有特征（包含新增的血压特征）
    x = tf.keras.layers.Concatenate()([x, age_in, gender_in, weight_in, height_in, 
                                      bsa_in, bmi_in, hr_in, sbp_in, dbp_in, pp_in])
    outputs = Dense(1, name="sv_output")(x)
    
    model = Model(
        inputs=[inputs, age_in, gender_in, weight_in, height_in, bsa_in, bmi_in, hr_in,
                sbp_in, dbp_in, pp_in],  # 包含新增的血压特征输入
        outputs=outputs
    )
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=best_hp["lr"], 
            clipvalue=1.0,
            beta_1=0.9,
            beta_2=0.999,
            epsilon=1e-07
        ),
        loss=tf.keras.losses.Huber(delta=1.0),
        metrics=['mae']
    )
    
    return model, best_hp

# ================== Model Evaluation ==================
def evaluate_final_model(model, X, y_true, hr, time_groups, dataset_name, save_dir):
    """Evaluate model and save results"""
    y_pred_sv = model.predict(X, verbose=0).flatten()
    
    if len(y_pred_sv) != len(y_true[:, 0]):
        min_length = min(len(y_pred_sv), len(y_true[:, 0]))
        y_pred_sv = y_pred_sv[:min_length]
        y_true = y_true[:min_length]
        hr = hr[:min_length]
        time_groups = time_groups[:min_length]
        logging.warning(f"Prediction and true value lengths do not match, truncated to {min_length} samples")
    
    y_pred_co = (y_pred_sv * hr) / 1000
    y_true_sv = y_true[:, 0]
    y_true_co = y_true[:, 1]
    
    def calculate_metrics(y_true, y_pred, name):
        mse = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)
        corr, p_val = pearsonr(y_true, y_pred)
        
        logging.info(f"\n{name} evaluation metrics:")
        logging.info(f"MSE: {mse:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f}")
        logging.info(f"R²: {r2:.4f} | Correlation coefficient: {corr:.4f} (p={p_val:.4e})")
        
        return {
            "mse": mse, "rmse": rmse, "mae": mae,
            "r2": r2, "corr": corr, "p_value": p_val,
            "y_true": y_true, "y_pred": y_pred
        }
    
    sv_metrics = calculate_metrics(y_true_sv, y_pred_sv, f"{dataset_name} - SV")
    co_metrics = calculate_metrics(y_true_co, y_pred_co, f"{dataset_name} - CO (calculated)")
    
    # 误差分布分析
    plt.figure(figsize=(12, 5))
    for i, (item, title) in enumerate([
        (sv_metrics, f"{dataset_name} SV Error Distribution"),
        (co_metrics, f"{dataset_name} CO Error Distribution")
    ]):
        plt.subplot(1, 2, i+1)
        error = item["y_pred"] - item["y_true"]
        plt.hist(error, bins=30, alpha=0.7)
        plt.axvline(x=0, color='r', linestyle='--')
        plt.xlabel("Prediction Error")
        plt.ylabel("Count")
        plt.title(title)
        plt.grid(ls='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(f"{save_dir}/{dataset_name}_error_distribution.png", dpi=300)
    plt.close()
    
    # 预测值vs真实值
    plt.figure(figsize=(12, 5))
    for i, (item, title) in enumerate([
        (sv_metrics, f"{dataset_name} SV Predictions vs True Values (R²={sv_metrics['r2']:.2f})"),
        (co_metrics, f"{dataset_name} CO Predictions vs True Values (R²={co_metrics['r2']:.2f})")
    ]):
        plt.subplot(1, 2, i+1)
        true_vals = item["y_true"]
        pred_vals = item["y_pred"]
        min_len = min(len(true_vals), len(pred_vals))
        
        plt.scatter(true_vals[:min_len], pred_vals[:min_len], alpha=0.6, s=30)
        min_val = min(true_vals[:min_len].min(), pred_vals[:min_len].min())
        max_val = max(true_vals[:min_len].max(), pred_vals[:min_len].max())
        plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1)
        plt.xlabel("True Value")
        plt.ylabel("Predicted Value")
        plt.title(title)
        plt.grid(ls='--', alpha=0.5)
    
    plt.tight_layout()
    plt.savefig(f"{save_dir}/{dataset_name}_pred_vs_true.png", dpi=300)
    plt.close()
    
    # 按时间组分析
    time_groups_unique = np.unique(time_groups)
    group_metrics = []
    for t in time_groups_unique:
        mask = time_groups == t
        if np.sum(mask) < 5:
            continue
        group_r2 = r2_score(y_true_sv[mask], y_pred_sv[mask])
        group_metrics.append({"time_group": t, "sample_count": np.sum(mask), "r2": group_r2})
    
    if group_metrics:
        pd.DataFrame(group_metrics).to_csv(f"{save_dir}/{dataset_name}_time_group_metrics.csv", index=False)
        logging.info(f"Saved time group metrics for {dataset_name} with {len(group_metrics)} groups")
    
    # 针对测试集添加Bland-Altman图
    if dataset_name == "Test set":
        analyze_grouped_predictions(y_true_sv, y_pred_sv, y_true_co, y_pred_co, save_dir)
        
        # 绘制并保存Bland-Altman图
        ba_sv_results = plot_bland_altman(
            y_true_sv, y_pred_sv, 
            metric_name="SV", 
            save_path=f"{save_dir}/test_bland_altman_sv.png"
        )
        ba_co_results = plot_bland_altman(
            y_true_co, y_pred_co, 
            metric_name="CO", 
            save_path=f"{save_dir}/test_bland_altman_co.png"
        )
        
        # 记录Bland-Altman统计结果
        logging.info("\nBland-Altman Analysis for Test Set:")
        logging.info(f"SV - Mean Bias: {ba_sv_results['mean_bias']:.4f}, 95% LoA: [{ba_sv_results['lower_limit']:.4f}, {ba_sv_results['upper_limit']:.4f}]")
        logging.info(f"CO - Mean Bias: {ba_co_results['mean_bias']:.4f}, 95% LoA: [{ba_co_results['lower_limit']:.4f}, {ba_co_results['upper_limit']:.4f}]")
    
    return {"SV": sv_metrics, "CO": co_metrics}

def analyze_grouped_predictions(y_true_sv, y_pred_sv, y_true_co, y_pred_co, save_dir):
    """Grouped analysis for test set"""
    df_sv = pd.DataFrame({'true_value': y_true_sv, 'pred_value': y_pred_sv})
    df_co = pd.DataFrame({'true_value': y_true_co, 'pred_value': y_pred_co})
    
    grouped_sv = df_sv.groupby(df_sv['true_value'].round(1)).agg({'pred_value': 'mean'}).reset_index()
    grouped_co = df_co.groupby(df_co['true_value'].round(1)).agg({'pred_value': 'mean'}).reset_index()
    
    def calculate_grouped_metrics(grouped_df, name):
        mse = mean_squared_error(grouped_df['true_value'], grouped_df['pred_value'])
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(grouped_df['true_value'], grouped_df['pred_value'])
        r2 = r2_score(grouped_df['true_value'], grouped_df['pred_value'])
        corr, p_val = pearsonr(grouped_df['true_value'], grouped_df['pred_value'])
        
        logging.info(f"\nGrouped {name} evaluation metrics:")
        logging.info(f"MSE: {mse:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f}")
        logging.info(f"R²: {r2:.4f} | Correlation coefficient: {corr:.4f} (p={p_val:.4e})")
        
        return {"mse": mse, "rmse": rmse, "mae": mae, "r2": r2, "corr": corr, "p_value": p_val, "grouped_data": grouped_df}
    
    grouped_sv_metrics = calculate_grouped_metrics(grouped_sv, "SV")
    grouped_co_metrics = calculate_grouped_metrics(grouped_co, "CO")
    
    grouped_sv.to_csv(f"{save_dir}/test_grouped_sv.csv", index=False)
    grouped_co.to_csv(f"{save_dir}/test_grouped_co.csv", index=False)
    
    plt.figure(figsize=(12, 5))
    for i, (grouped_data, title, metric) in enumerate([
        (grouped_sv, f"Test Set Grouped SV Prediction Mean vs True Values (R²={grouped_sv_metrics['r2']:.2f})", grouped_sv_metrics),
        (grouped_co, f"Test Set Grouped CO Prediction Mean vs True Values (R²={grouped_co_metrics['r2']:.2f})", grouped_co_metrics)
    ]):
        plt.subplot(1, 2, i+1)
        plt.scatter(grouped_data['true_value'], grouped_data['pred_value'], alpha=0.7, s=50, c='green')
        min_val = min(grouped_data['true_value'].min(), grouped_data['pred_value'].min())
        max_val = max(grouped_data['true_value'].max(), grouped_data['pred_value'].max())
        plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1)
        
        plt.text(0.05, 0.95, 
                 f"r = {metric['corr']:.3f}\nn = {len(grouped_data)}", 
                 transform=plt.gca().transAxes,
                 verticalalignment='top',
                 bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        plt.xlabel("True Value")
        plt.ylabel("Mean Predicted Value")
        plt.title(title)
        plt.grid(ls='--', alpha=0.5)
    
    plt.tight_layout()
    plt.savefig(f"{save_dir}/test_grouped_pred_vs_true.png", dpi=300)
    plt.close()

# ================== 新增：将模型导出为ONNX格式 ==================
def export_model_to_onnx(model, save_path):
    """
    将Keras模型导出为ONNX格式 - 包含新增的血压特征输入
    """
    try:
        # 定义输入形状（批量大小设为None以支持动态批量）
        input_spec = (
            tf.TensorSpec((None, 125, 3), tf.float32, name="signal_input"),
            tf.TensorSpec((None,), tf.float32, name="age_input"),
            tf.TensorSpec((None,), tf.float32, name="gender_input"),
            tf.TensorSpec((None,), tf.float32, name="weight_input"),
            tf.TensorSpec((None,), tf.float32, name="height_input"),
            tf.TensorSpec((None,), tf.float32, name="bsa_input"),
            tf.TensorSpec((None,), tf.float32, name="bmi_input"),
            tf.TensorSpec((None,), tf.float32, name="hr_input"),
            tf.TensorSpec((None,), tf.float32, name="sbp_input"),  # 新增SBP输入
            tf.TensorSpec((None,), tf.float32, name="dbp_input"),  # 新增DBP输入
            tf.TensorSpec((None,), tf.float32, name="pp_input")    # 新增PP输入
        )
        
        # 转换为ONNX
        onnx_model, _ = tf2onnx.convert.from_keras(model, input_signature=input_spec, opset=13)
        
        # 保存ONNX模型
        onnx.save(onnx_model, save_path)
        
        # 验证模型是否可加载
        session = InferenceSession(save_path)
        logging.info(f"模型成功导出为ONNX格式: {save_path}")
        return True
    except Exception as e:
        logging.error(f"ONNX模型导出失败: {str(e)}")
        return False

# ================== Main Function ==================
def main():
    set_seed()
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    save_dir = f"autodl-tmp/cnap_1s_sv/final_model1012/resnet_SE_LSTM/{timestamp}"
    os.makedirs(save_dir, exist_ok=True)
    logging.info(f"All results will be saved to: {save_dir}")
    
    # 1. 读取数据
    file_path = "autodl-tmp/cnap_1s_sv/数据/总711-1112合并后的血压数据.csv"
    df = read_csv_with_encoding(file_path)
    if df is None or len(df) == 0:
        logging.error("Could not read valid data, program terminated")
        return
    
    # 2. 提取特征
    try:
        # 新增SBP、DBP、PP特征的提取
        X, y, age, gender, weight, height, bsa, bmi, hr, sbp, dbp, pp, time, indices, original_df = extract_data(df)
    except Exception as e:
        logging.error(f"Data extraction failed: {e}")
        return
    
    # 异常值处理
    sv_values = y[:, 0]
    q1, q3 = np.percentile(sv_values, [15, 85])
    iqr = q3 - q1
    upper_bound = q3 + 1.5 * iqr
    valid_mask = sv_values <= upper_bound
    
    X = X[valid_mask]
    y = y[valid_mask]
    age = age[valid_mask]
    gender = gender[valid_mask]
    weight = weight[valid_mask]
    height = height[valid_mask]
    bsa = bsa[valid_mask]
    bmi = bmi[valid_mask]
    hr = hr[valid_mask]
    sbp = sbp[valid_mask]  # 新增SBP异常值过滤
    dbp = dbp[valid_mask]  # 新增DBP异常值过滤
    pp = pp[valid_mask]    # 新增PP异常值过滤
    time = time[valid_mask]
    indices = indices[valid_mask]
    original_df = original_df[valid_mask].copy()
    
    removed_count = len(valid_mask) - sum(valid_mask)
    logging.info(f"使用IQR法检测异常值，上界={upper_bound:.2f}，移除{removed_count}个样本，剩余有效样本: {len(X)}")
    
    if len(X) < 100:
        logging.error(f"Insufficient data volume after filtering (only {len(X)} samples), cannot train model")
        return
    
    # 3. 划分数据集
    try:
        # 新增SBP、DBP、PP的数据集划分
        split_result = split_data_by_time_groups(
            X, y, age, gender, weight, height, bsa, bmi, hr, sbp, dbp, pp, y[:, 0], time, indices
        )
        (X_train, X_val, X_test, y_train, y_val, y_test,
         age_train, age_val, age_test, gender_train, gender_val, gender_test,
         weight_train, weight_val, weight_test, height_train, height_val, height_test,
         bsa_train, bsa_val, bsa_test, bmi_train, bmi_val, bmi_test,
         hr_train, hr_val, hr_test, sbp_train, sbp_val, sbp_test,
         dbp_train, dbp_val, dbp_test, pp_train, pp_val, pp_test,
         time_train, time_val, time_test, indices_train, indices_val, indices_test) = split_result
    except Exception as e:
        logging.error(f"Dataset splitting failed: {e}")
        return
    
    # 训练集数据增强
    X_train_augmented = np.array([augment_signal(x) for x in X_train])
    logging.info(f"Applied data augmentation to training set: {X_train_augmented.shape}")
    
    # 4. 标准化特征
    logging.info("Standardizing features...")
    # 1. 信号特征标准化器
    X_train_std, X_val_std, X_test_std, x_scaler = standardize_features(X_train_augmented, X_val, X_test, is_2d=True)
    
    # 2. 临床特征标准化器（新增血压特征的标准化）
    age_train_std, age_val_std, age_test_std, age_scaler = standardize_features(age_train, age_val, age_test)
    gender_train_std, gender_val_std, gender_test_std = gender_train, gender_val, gender_test
    gender_scaler = None
    weight_train_std, weight_val_std, weight_test_std, weight_scaler = standardize_features(weight_train, weight_val, weight_test)
    height_train_std, height_val_std, height_test_std, height_scaler = standardize_features(height_train, height_val, height_test)
    bsa_train_std, bsa_val_std, bsa_test_std, bsa_scaler = standardize_features(bsa_train, bsa_val, bsa_test)
    bmi_train_std, bmi_val_std, bmi_test_std, bmi_scaler = standardize_features(bmi_train, bmi_val, bmi_test)
    hr_train_std, hr_val_std, hr_test_std, hr_scaler = standardize_features(hr_train, hr_val, hr_test)
    sbp_train_std, sbp_val_std, sbp_test_std, sbp_scaler = standardize_features(sbp_train, sbp_val, sbp_test)  # 新增SBP标准化
    dbp_train_std, dbp_val_std, dbp_test_std, dbp_scaler = standardize_features(dbp_train, dbp_val, dbp_test)  # 新增DBP标准化
    pp_train_std, pp_val_std, pp_test_std, pp_scaler = standardize_features(pp_train, pp_val, pp_test)        # 新增PP标准化
    
    # 保存所有特征的标准化器
    joblib.dump(x_scaler, f"{save_dir}/signal_scaler.pkl")
    joblib.dump(age_scaler, f"{save_dir}/age_scaler.pkl")
    joblib.dump(weight_scaler, f"{save_dir}/weight_scaler.pkl")
    joblib.dump(height_scaler, f"{save_dir}/height_scaler.pkl")
    joblib.dump(bsa_scaler, f"{save_dir}/bsa_scaler.pkl")
    joblib.dump(bmi_scaler, f"{save_dir}/bmi_scaler.pkl")
    joblib.dump(hr_scaler, f"{save_dir}/hr_scaler.pkl")
    joblib.dump(sbp_scaler, f"{save_dir}/sbp_scaler.pkl")  # 新增SBP标准化器
    joblib.dump(dbp_scaler, f"{save_dir}/dbp_scaler.pkl")  # 新增DBP标准化器
    joblib.dump(pp_scaler, f"{save_dir}/pp_scaler.pkl")    # 新增PP标准化器
    
    logging.info("All feature scalers saved successfully")
    
    # 5. 构建模型 - 先ResNet+SE，再LSTM
    logging.info("Building model with ResNet+SE first, then LSTM...")
    model, best_hp = build_resnet18_with_se_lstm(input_shape=(125, 3))
    model.summary(print_fn=lambda x: logging.info(x))
    
    with open(f"{save_dir}/best_hyperparameters.txt", "w") as f:
        for key, value in best_hp.items():
            f.write(f"{key}: {value}\n")
    
    # 6. 训练模型
    logging.info("Starting model training...")
    batch_size = 32 if len(X_train) < 1000 else 64
    logging.info(f"Using batch size: {batch_size} based on training set size")
    
    def cosine_annealing(epoch, lr):
        initial_lr = best_hp["lr"]
        epochs = 50
        return initial_lr * (1 + np.cos(np.pi * epoch / epochs)) / 2
    
    callbacks = [
        LearningRateScheduler(cosine_annealing, verbose=1)
    ]
    
    # 训练数据中加入新增的血压特征
    history = model.fit(
        [X_train_std, age_train_std, gender_train_std, weight_train_std, 
         height_train_std, bsa_train_std, bmi_train_std, hr_train_std,
         sbp_train_std, dbp_train_std, pp_train_std],  # 新增血压特征
        y_train[:, 0],
        epochs=50,
        batch_size=batch_size,
        validation_data=(
            [X_val_std, age_val_std, gender_val_std, weight_val_std, 
             height_val_std, bsa_val_std, bmi_val_std, hr_val_std,
             sbp_val_std, dbp_val_std, pp_val_std],  # 新增血压特征
            y_val[:, 0]
        ),
        callbacks=callbacks,
        verbose=1
    )
    
    pd.DataFrame(history.history).to_csv(f"{save_dir}/training_history.csv", index=False)
    
    # 绘制训练曲线
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Loss Curves')
    plt.xlabel('Epoch')
    plt.ylabel('Huber Loss')
    plt.legend()
    plt.grid(ls='--', alpha=0.5)
    
    plt.subplot(1, 2, 2)
    plt.plot(history.history['mae'], label='Training MAE')
    plt.plot(history.history['val_mae'], label='Validation MAE')
    plt.title('MAE Curves')
    plt.xlabel('Epoch')
    plt.ylabel('MAE')
    plt.legend()
    plt.grid(ls='--', alpha=0.5)
    
    plt.tight_layout()
    plt.savefig(f"{save_dir}/training_curves.png", dpi=300)
    plt.close()
    
    # 7. 生成预测结果
    logging.info("Generating predictions for all datasets...")
    # 训练集预测（加入血压特征）
    pred_sv_train, pred_co_train = generate_predictions(
        model, X_train_std, age_train_std, gender_train_std, weight_train_std, 
        height_train_std, bsa_train_std, bmi_train_std, hr_train_std,
        sbp_train_std, dbp_train_std, pp_train_std,  # 新增血压特征
        hr_train
    )
    
    # 验证集预测（加入血压特征）
    pred_sv_val, pred_co_val = generate_predictions(
        model, X_val_std, age_val_std, gender_val_std, weight_val_std, 
        height_val_std, bsa_val_std, bmi_val_std, hr_val_std,
        sbp_val_std, dbp_val_std, pp_val_std,  # 新增血压特征
        hr_val
    )
    
    # 测试集预测（加入血压特征）
    pred_sv_test, pred_co_test = generate_predictions(
        model, X_test_std, age_test_std, gender_test_std, weight_test_std, 
        height_test_std, bsa_test_std, bmi_test_std, hr_test_std,
        sbp_test_std, dbp_test_std, pp_test_std,  # 新增血压特征
        hr_test
    )
    
    # 8. 保存包含预测结果的数据集
    save_datasets_with_predictions(
        save_dir,
        X_train, X_val, X_test, y_train, y_val, y_test,
        age_train, age_val, age_test, gender_train, gender_val, gender_test,
        weight_train, weight_val, weight_test, height_train, height_val, height_test,
        bsa_train, bsa_val, bsa_test, bmi_train, bmi_val, bmi_test,
        hr_train, hr_val, hr_test, sbp_train, sbp_val, sbp_test,
        dbp_train, dbp_val, dbp_test, pp_train, pp_val, pp_test,
        time_train, time_val, time_test, indices_train, indices_val, indices_test,
        pred_sv_train, pred_co_train,
        pred_sv_val, pred_co_val,
        pred_sv_test, pred_co_test
    )
    
    # 9. 评估模型
    logging.info("\n===== Starting model evaluation =====")
    # 测试集数据（加入血压特征）
    test_data = [X_test_std, age_test_std, gender_test_std, weight_test_std, 
                 height_test_std, bsa_test_std, bmi_test_std, hr_test_std,
                 sbp_test_std, dbp_test_std, pp_test_std]  # 新增血压特征
    # 验证集数据（加入血压特征）
    val_data = [X_val_std, age_val_std, gender_val_std, weight_val_std, 
                height_val_std, bsa_val_std, bmi_val_std, hr_val_std,
                sbp_val_std, dbp_val_std, pp_val_std]  # 新增血压特征
    # 训练集数据（加入血压特征）
    train_data = [X_train_std, age_train_std, gender_train_std, weight_train_std, 
                  height_train_std, bsa_train_std, bmi_train_std, hr_train_std,
                  sbp_train_std, dbp_train_std, pp_train_std]  # 新增血压特征
    
    evaluate_final_model(model, train_data, y_train, hr_train, time_train, "Training set", save_dir)
    evaluate_final_model(model, val_data, y_val, hr_val, time_val, "Validation set", save_dir)
    test_metrics = evaluate_final_model(model, test_data, y_test, hr_test, time_test, "Test set", save_dir)
    
    # 10. 保存模型（同时导出为ONNX格式）
    model.save(f"{save_dir}/resnet_se_lstm_model")
    logging.info(f"ResNet+SE then LSTM model saved to: {save_dir}/resnet_se_lstm_model")
    
    # 导出ONNX格式
    onnx_path = f"{save_dir}/resnet_se_lstm_model.onnx"
    export_success = export_model_to_onnx(model, onnx_path)
    if export_success:
        logging.info(f"ONNX model saved to: {onnx_path}")
    
    logging.info("\nAll processes completed!")

if __name__ == "__main__":
    main()


2025-11-13 17:10:35,079 - INFO - All results will be saved to: autodl-tmp/cnap_1s_sv/final_model1012/resnet_SE_LSTM/20251113_171035
/tmp/ipykernel_2169/3913192676.py:715: DtypeWarning: Columns (2,16) have mixed types.Specify dtype option on import or set low_memory=False.
  df = read_csv_with_encoding(file_path)
2025-11-13 17:10:35,750 - INFO - Successfully read file using encoding: utf-8 (sep=',' engine='c')
2025-11-13 17:10:35,998 - INFO - Feature extraction completed - Input feature shape: (47584, 125, 3)
2025-11-13 17:10:36,233 - INFO - 使用IQR法检测异常值，上界=151.00，移除40个样本，剩余有效样本: 47544
2025-11-13 17:10:36,250 - INFO - Detected 123 time groups
2025-11-13 17:10:36,349 - INFO - Dataset split completed - Training set: 29304 samples, Validation set: 7297 samples, Test set: 10943 samples
2025-11-13 17:10:36,350 - INFO - Training set contains 78 time groups, Validation set contains 20 time groups, Test set contains 25 time groups
2025-11-13 17:10:36,992 - INFO - Applied data augmentation to tra

Epoch 1/50

Epoch 00001: LearningRateScheduler reducing learning rate to 0.001.
458/458 [==============================] - 28s 51ms/step - loss: 20.7094 - mae: 21.0513 - val_loss: 12.9541 - val_mae: 13.2780
Epoch 2/50

Epoch 00002: LearningRateScheduler reducing learning rate to 0.0009990133642141358.
458/458 [==============================] - 23s 49ms/step - loss: 10.5579 - mae: 10.8628 - val_loss: 11.9563 - val_mae: 12.2545
Epoch 3/50

Epoch 00003: LearningRateScheduler reducing learning rate to 0.000996057350657239.
458/458 [==============================] - 21s 47ms/step - loss: 9.1616 - mae: 9.4486 - val_loss: 11.7987 - val_mae: 12.0838
Epoch 4/50

Epoch 00004: LearningRateScheduler reducing learning rate to 0.0009911436253643444.
458/458 [==============================] - 22s 48ms/step - loss: 8.5337 - mae: 8.8064 - val_loss: 11.6220 - val_mae: 11.8889
Epoch 5/50

Epoch 00005: LearningRateScheduler reducing learning rate to 0.0009842915805643156.
458/458 [========================

2025-11-13 17:29:06,284 - INFO - Generating predictions for all datasets...
2025-11-13 17:29:20,837 - INFO - Datasets with predictions saved to: autodl-tmp/cnap_1s_sv/final_model1012/resnet_SE_LSTM/20251113_171035/datasets
2025-11-13 17:29:20,838 - INFO - 
===== Starting model evaluation =====
2025-11-13 17:29:28,982 - INFO - 
Training set - SV evaluation metrics:
2025-11-13 17:29:28,983 - INFO - MSE: 8.6121 | RMSE: 2.9346 | MAE: 1.9774
2025-11-13 17:29:28,984 - INFO - R²: 0.9762 | Correlation coefficient: 0.9886 (p=0.0000e+00)
2025-11-13 17:29:28,996 - INFO - 
Training set - CO (calculated) evaluation metrics:
2025-11-13 17:29:28,997 - INFO - MSE: 0.0489 | RMSE: 0.2210 | MAE: 0.1476
2025-11-13 17:29:28,997 - INFO - R²: 0.9807 | Correlation coefficient: 0.9908 (p=0.0000e+00)
2025-11-13 17:29:30,878 - INFO - Saved time group metrics for Training set with 78 groups
2025-11-13 17:29:33,026 - INFO - 
Validation set - SV evaluation metrics:
2025-11-13 17:29:33,027 - INFO - MSE: 199.1383 | R

INFO:tensorflow:Assets written to: autodl-tmp/cnap_1s_sv/final_model1012/resnet_SE_LSTM/20251113_171035/resnet_se_lstm_model/assets


2025-11-13 17:29:50,392 - INFO - Assets written to: autodl-tmp/cnap_1s_sv/final_model1012/resnet_SE_LSTM/20251113_171035/resnet_se_lstm_model/assets
2025-11-13 17:29:50,716 - INFO - ResNet+SE then LSTM model saved to: autodl-tmp/cnap_1s_sv/final_model1012/resnet_SE_LSTM/20251113_171035/resnet_se_lstm_model
2025-11-13 17:29:51.034082: I tensorflow/core/grappler/devices.cc:69] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
2025-11-13 17:29:51.034242: I tensorflow/core/grappler/clusters/single_machine.cc:357] Starting new session
2025-11-13 17:29:51.035699: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1733] Found device 0 with properties: 
pciBusID: 0000:c8:00.0 name: NVIDIA GeForce RTX 4090 computeCapability: 8.9
coreClock: 2.52GHz coreCount: 128 deviceMemorySize: 47.38GiB deviceMemoryBandwidth: 938.86GiB/s
2025-11-13 17:29:51.037560: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1871] Adding visible gpu devices: 0
2025-11-13 17:29:51.037591: I tensorf

Instructions for updating:
Use `tf.compat.v1.graph_util.extract_sub_graph`


2025-11-13 17:29:51.435899: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1733] Found device 0 with properties: 
pciBusID: 0000:c8:00.0 name: NVIDIA GeForce RTX 4090 computeCapability: 8.9
coreClock: 2.52GHz coreCount: 128 deviceMemorySize: 47.38GiB deviceMemoryBandwidth: 938.86GiB/s
2025-11-13 17:29:51.437928: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1871] Adding visible gpu devices: 0
2025-11-13 17:29:51.437959: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1258] Device interconnect StreamExecutor with strength 1 edge matrix:
2025-11-13 17:29:51.437962: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1264]      0 
2025-11-13 17:29:51.437965: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1277] 0:   N 
2025-11-13 17:29:51.440074: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1418] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 46440 MB memory) -> physical GPU (device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:c8:00